# ROME: Hierarchical RTL Generation and Coverage Refinement

Model: OpenAI gpt-4o. Provider: ChatGPT. Colab secret: ROME.

Executed results are preserved below. Run in a fresh Colab runtime for a new generation experiment; cells that create fixed baseline testbenches intentionally reject overwriting existing files. API generation is nondeterministic.

Final instance coverage: line 40/40; toggle 189/189 in each direction; logic 136/136. Final testbench: 31 passing tests.

See ROME_Report.pdf and README.md for commands and evidence. The duplicate legacy API configuration and duplicate testbench download cell were removed during submission cleanup.

# Initial Setup

In [1]:
#@title Setting up the notebook

### Installing dependencies
!pip install openai
!pip install anthropic
!apt-get update
!apt-get install -y iverilog verilator gtkwave

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 10.4 MB/s  0:00:00
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Get:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble/main amd64 Packages [60.0 kB]
Get:10 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,549 kB]
Get:11 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu noble/main amd64 Packages [3,088 kB]
Ge

In [3]:
import shutil
import subprocess

for tool in ["iverilog", "vvp"]:
    path = shutil.which(tool)
    print(f"{tool}: {path}")
    assert path is not None, f"{tool} "

result = subprocess.run(
    ["iverilog", "-V"],
    capture_output=True,
    text=True
)

print("\n".join(result.stdout.splitlines()[:3]))

iverilog: /usr/bin/iverilog
vvp: /usr/bin/vvp
Icarus Verilog version 12.0 (stable) ()

Copyright (c) 2000-2021 Stephen Williams (steve@icarus.com)


In [4]:
#@title Setting up PPA estimation

%%bash
set -e
cd /content

URL=$(python3 - <<'PY'
import json, urllib.request
data = json.load(urllib.request.urlopen(
    "https://api.github.com/repos/YosysHQ/oss-cad-suite-build/releases/latest"
))
asset = next(
    a for a in data["assets"]
    if "linux-x64" in a["name"] and a["name"].endswith(".tgz")
)
print(asset["browser_download_url"])
PY
)

wget -q -O oss-cad-suite.tgz "$URL"
rm -rf /content/oss-cad-suite /content/oss-cad-suite-install
mkdir -p /content/oss-cad-suite-install
tar -xzf oss-cad-suite.tgz -C /content/oss-cad-suite-install
mv /content/oss-cad-suite-install/oss-cad-suite /content/oss-cad-suite

/content/oss-cad-suite/bin/yosys -V



Yosys 0.70+219 (git sha1 5053d0e26, Release, Clang /usr/bin/clang++ 21.1.8)


In [5]:
#@title Select Model
#define the model to be used
#model_choice = "gpt-5.2"
#model_choice = "gpt-5.2-codex"
model_choice = "gpt-4o"
#model_choice = "claude-sonnet-4-6"
#model_choice = "gemini-2.5-flash-preview-04-17"
#model_choice = "gemini-2.5-flash"
#model_choice = "nyu-dice-lab/VeriThoughts-Reasoning-14B" #path to your desired Hugging Face model

In [6]:
import os
from google.colab import userdata

api_key = userdata.get("ROME")
assert api_key and api_key.strip(), "ROME secret is empty"

os.environ["OPENAI_API_KEY"] = api_key
os.environ["MODEL"] = "ChatGPT"

print("API key loaded from ROME")
print("Provider:", os.environ["MODEL"])
print("Model:", model_choice)

API key loaded from ROME
Provider: ChatGPT
Model: gpt-4o


In [7]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model=model_choice,
    messages=[
        {"role": "user", "content": "Reply with exactly: API connection successful"}
    ]
)

print(response.choices[0].message.content)

API connection successful


In [8]:
#@title Utility functions

import sys
import os
import time
import re
from abc import ABC, abstractmethod

import openai
import anthropic

try:
    import google.genai.errors
    from google import genai
    from google.genai import types
except Exception:
    genai = None
    types = None

################################################################################
### LOGGING
################################################################################

class LogStdoutToFile:
    def __init__(self, filename):
        self._filename = filename
        self._original_stdout = sys.stdout

    def __enter__(self):
        if self._filename:
            sys.stdout = open(self._filename, 'w')
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self._filename:
            sys.stdout.close()
        sys.stdout = self._original_stdout


################################################################################
### CONVERSATION CLASS
################################################################################

class Conversation:
    def __init__(self, log_file=None):
        self.messages = []
        self.log_file = log_file

        if self.log_file and os.path.exists(self.log_file):
            open(self.log_file, 'w').close()

    def add_message(self, role, content):
        self.messages.append({'role': role, 'content': content})

        if self.log_file:
            with open(self.log_file, 'a') as file:
                file.write(f"{role}: {content}\n")

    def get_messages(self):
        return self.messages

    def get_last_n_messages(self, n):
        return self.messages[-n:]

    def remove_message(self, index):
        if index < len(self.messages):
            del self.messages[index]

    def get_message(self, index):
        return self.messages[index] if index < len(self.messages) else None

    def clear_messages(self):
        self.messages = []

    def __str__(self):
        return "\n".join([f"{msg['role']}: {msg['content']}" for msg in self.messages])


################################################################################
### LLM CLASSES
################################################################################

class AbstractLLM(ABC):
    @abstractmethod
    def generate(self, conversation: Conversation):
        pass


class ChatGPT(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        openai.api_key = os.environ["OPENAI_API_KEY"]
        self.client = openai.OpenAI()
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        messages = [
            {
                "role": msg["role"] if msg["role"] in ["system", "user", "assistant"] else "user",
                "content": msg["content"]
            }
            for msg in conversation.get_messages()
        ]

        response = self.client.chat.completions.create(
            model=self.model_id,
            messages=messages,
        )

        return response.choices[0].message.content


class Claude(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        self.client = anthropic.Anthropic(api_key=os.environ["CLAUDE_API_KEY"])
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        base_delay = 2
        max_retries = 20

        messages = [
            {
                "role": msg["role"] if msg["role"] in ["user", "assistant"] else "user",
                "content": msg["content"]
            }
            for msg in conversation.get_messages()
        ]

        for attempt in range(1, max_retries + 1):
            try:
                output = self.client.messages.create(
                    model=self.model_id,
                    max_tokens=16384,
                    messages=messages
                ).content[0].text
                return output

            except Exception as e:
                wait_time = base_delay * (2 ** (attempt - 1))
                print(f"[Retry {attempt}/{max_retries}] Claude API error: {e}. Retrying in {wait_time:.1f} seconds...")
                time.sleep(wait_time)

        print(f"Failed, exceeded max retries {max_retries}")
        return ""


class Gemini(AbstractLLM):
    def __init__(self, model_id=None):
        super().__init__()
        if genai is None:
            raise ImportError("google-genai is not installed or could not be imported.")
        self.gemini_client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
        self.model_id = model_id or model_choice

    def generate(self, conversation: Conversation, num_choices=1):
        output = self.gemini_client.models.generate_content(
            model=self.model_id,
            contents=[msg["content"] for msg in conversation.get_messages()],
            config=types.GenerateContentConfig(
                max_output_tokens=16384,
                temperature=0.6,
                topP=0.95,
            )
        ).text

        return output


class HuggingFace(AbstractLLM):
    """
    Local Hugging Face model backend.

    Downloads the selected model from the Hugging Face Hub using snapshot_download,
    then loads it with transformers. The model is cached and reused so it is not
    reloaded every time generate_verilog() is called.
    """

    _loaded_model_id = None
    _tokenizer = None
    _model = None
    _local_model_path = None

    def __init__(
        self,
        model_id=None,
        max_new_tokens=4096,
        temperature=0.2,
        top_p=0.95,
        load_in_4bit=False,
        trust_remote_code=True,
        strip_thinking=True,
    ):
        super().__init__()

        self.model_id = model_id or model_choice

        # If the user accidentally wrote "/org/model", convert it to "org/model".
        # Absolute local paths like "/content/my_model" are still allowed.
        if self.model_id.startswith("/") and not self.model_id.startswith("/content/"):
            self.model_id = self.model_id[1:]

        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.top_p = top_p
        self.load_in_4bit = load_in_4bit
        self.trust_remote_code = trust_remote_code
        self.strip_thinking = strip_thinking

        self.hf_token = (
            os.environ.get("HF_TOKEN")
            or os.environ.get("HUGGINGFACEHUB_API_TOKEN")
            or None
        )

        self.cache_dir = os.environ.get("HF_HOME", "/content/huggingface")

        self._ensure_model_loaded()

    def _get_torch_dtype(self):
        import torch

        if not torch.cuda.is_available():
            return torch.float32

        major, minor = torch.cuda.get_device_capability(0)

        # A100/H100 and newer generally support bf16 well.
        if major >= 8:
            return torch.bfloat16

        # T4/V100 are safer with fp16.
        return torch.float16

    def _ensure_model_loaded(self):
        if HuggingFace._model is not None and HuggingFace._loaded_model_id == self.model_id:
            return

        import torch
        from huggingface_hub import snapshot_download
        from transformers import AutoTokenizer, AutoModelForCausalLM

        print(f"[HuggingFace] Downloading/caching model: {self.model_id}")

        # If this is a local path, do not call snapshot_download.
        if os.path.isdir(self.model_id):
            local_model_path = self.model_id
        else:
            local_model_path = snapshot_download(
                repo_id=self.model_id,
                cache_dir=self.cache_dir,
                token=self.hf_token,
            )

        print(f"[HuggingFace] Loading tokenizer from: {local_model_path}")

        tokenizer = AutoTokenizer.from_pretrained(
            local_model_path,
            trust_remote_code=self.trust_remote_code,
            token=self.hf_token,
        )

        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token

        torch_dtype = self._get_torch_dtype()

        model_kwargs = {
            "trust_remote_code": self.trust_remote_code,
            "device_map": "auto",
            "low_cpu_mem_usage": True,
            "token": self.hf_token,
        }

        if torch.cuda.is_available() and self.load_in_4bit:
            from transformers import BitsAndBytesConfig

            quantization_config = BitsAndBytesConfig(
                load_in_4bit=False,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch_dtype,
                bnb_4bit_use_double_quant=True,
            )

            model_kwargs["quantization_config"] = quantization_config
        else:
            model_kwargs["torch_dtype"] = torch_dtype

        print("[HuggingFace] Loading model. This can take a few minutes the first time.")

        model = AutoModelForCausalLM.from_pretrained(
            local_model_path,
            **model_kwargs
        )

        model.eval()

        HuggingFace._loaded_model_id = self.model_id
        HuggingFace._tokenizer = tokenizer
        HuggingFace._model = model
        HuggingFace._local_model_path = local_model_path

        print("[HuggingFace] Model loaded successfully.")

    def _format_prompt(self, conversation: Conversation):
        tokenizer = HuggingFace._tokenizer

        messages = []
        for msg in conversation.get_messages():
            role = msg.get("role", "user")
            content = msg.get("content", "")

            if role not in ["system", "user", "assistant"]:
                role = "user"

            messages.append({"role": role, "content": content})

        # Prefer the model's own chat template when available.
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
        except Exception:
            # Generic fallback for non-chat-template models.
            prompt_parts = []
            for msg in messages:
                prompt_parts.append(f"{msg['role'].upper()}:\n{msg['content']}\n")
            prompt_parts.append("ASSISTANT:\n")
            return "\n".join(prompt_parts)

    def _clean_output(self, text):
        text = text.strip()

        # Helpful for reasoning models that emit <think>...</think>.
        if self.strip_thinking:
            text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

        return text

    def generate(self, conversation: Conversation, num_choices=1):
        import torch

        tokenizer = HuggingFace._tokenizer
        model = HuggingFace._model

        prompt = self._format_prompt(conversation)

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=False,
        )

        # With device_map="auto", use the first model parameter's device.
        first_device = next(model.parameters()).device
        inputs = {k: v.to(first_device) for k, v in inputs.items()}

        do_sample = self.temperature is not None and self.temperature > 0

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=do_sample,
                temperature=self.temperature if do_sample else None,
                top_p=self.top_p if do_sample else None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_len = inputs["input_ids"].shape[-1]
        generated_ids = outputs[0][prompt_len:]

        text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return self._clean_output(text)


################################################################################
### PARSING AND TEXT MANIPULATION FUNCTIONS
################################################################################

def find_verilog_modules(markdown_string, module_name='top_module'):
    module_pattern1 = r'\bmodule\b\s+\w+\s*\([^)]*\)\s*;.*?endmodule\b'
    module_pattern2 = r'\bmodule\b\s+\w+\s*#\s*\([^)]*\)\s*\([^)]*\)\s*;.*?endmodule\b'

    module_matches1 = re.findall(module_pattern1, markdown_string, re.DOTALL)
    module_matches2 = re.findall(module_pattern2, markdown_string, re.DOTALL)

    module_matches = module_matches1 + module_matches2

    if not module_matches:
        return []

    return module_matches


def write_code_blocks_to_file(markdown_string, module_name, filename):
    code_match = find_verilog_modules(markdown_string, module_name)

    if not code_match:
        print("No Verilog modules found in response")
        exit(3)

    with open(filename, 'w') as file:
        for code_block in code_match:
            file.write(code_block)
            file.write('\n')


def generate_verilog(conv, model_type=None, model_id=""):
    model_type = model_type or os.environ.get("MODEL", "ChatGPT")
    selected_model_id = model_id or model_choice

    if model_type == "ChatGPT":
        model = ChatGPT(model_id=selected_model_id)
    elif model_type == "Claude":
        model = Claude(model_id=selected_model_id)
    elif model_type == "Gemini":
        model = Gemini(model_id=selected_model_id)
    elif model_type == "HuggingFace":
        model = HuggingFace(model_id=selected_model_id)
    else:
        raise ValueError(
            f"Invalid model type: {model_type}. "
            "Expected one of: ChatGPT, Claude, Gemini, HuggingFace."
        )

    return model.generate(conv)

In [9]:
#@title Feedback Loop

import subprocess
import sys
import os
import time
import numpy as np


def _read_files(file_list):
    """Concatenate a list of text files in order. Missing/None entries are ignored."""
    if not file_list:
        return ""

    buf = []
    for p in file_list:
        if not p:
            continue

        try:
            with open(p, "r") as f:
                text = f.read()
                buf.append(text)
                if not text.endswith("\n"):
                    buf.append("\n")
        except FileNotFoundError:
            print(f"[Warning] Previous module file not found and will be skipped: {p}")

    return "".join(buf)


def _truncate_text(s, max_chars=12000):
    """Keep prompts bounded; preserve the end of long logs, where compiler errors usually appear."""
    if s is None:
        return ""

    s = str(s)

    if len(s) <= max_chars:
        return s

    tail = s[-max_chars:]
    return "[...truncated...]\n" + tail


def _add_instruction_prompt(conv, model_type, base_sys_prompt):
    """
    Add the instruction prompt in a way that works for API models and local HF models.

    ChatGPT/Gemini/HuggingFace can use a system role.
    Claude path in the earlier utility cell maps unsupported roles to user,
    but we keep Claude as user here to avoid role issues.
    """
    if model_type == "Claude":
        conv.add_message("user", base_sys_prompt)
    else:
        conv.add_message("system", base_sys_prompt)


def _simulation_passed(stdout_text, stderr_text=""):
    """
    Robust pass/fail detection for your Verilog testbenches.

    Accepts:
      - "All tests passed"
      - "passed!"
      - "Test x passed"

    Rejects:
      - "failed"
      - "Some tests failed"
      - non-empty fatal/error-like stderr
    """
    out = (stdout_text or "")
    err = (stderr_text or "")
    combined = (out + "\n" + err).lower()

    fail_markers = [
        "failed",
        "some tests failed",
        "error running",
        "fatal",
        "syntax error",
    ]

    if any(marker in combined for marker in fail_markers):
        return False

    pass_markers = [
        "all tests passed",
        "passed!",
        "test 1 passed",
        " passed",
    ]

    return any(marker in combined for marker in pass_markers)


def verilog_loop(
    design_prompt,
    module,
    testbench,
    max_iterations,
    model_type=None,
    outdir="",
    log=None,
    prev_modules=None,
    model_id="",
):
    """
    Generate/fix a Verilog module with an LLM, compiling against previously-generated modules.

    Compatible model_type values:
      - "ChatGPT"
      - "Claude"
      - "Gemini"
      - "HuggingFace"

    Key behaviors:
      - The LLM is asked to output ONLY the current module.
      - For compilation, this concatenates:
            [prev_modules...] + [current module]
        into a temporary combined file.
      - After an error, repair iterations only resend:
            (a) previous module version
            (b) most recent compiler/simulation feedback
        to keep prompt size controlled.
    """

    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_model_types = ["ChatGPT", "Claude", "Gemini", "HuggingFace"]
    if model_type not in valid_model_types:
        raise ValueError(
            f"Invalid model_type: {model_type}. "
            f"Expected one of: {valid_model_types}"
        )

    if outdir is None:
        outdir = ""

    outdir = outdir.strip()

    if outdir != "":
        os.makedirs(outdir, exist_ok=True)

    def _outpath(filename):
        if outdir == "":
            return filename
        return os.path.join(outdir, filename)

    # Normalize prev_modules into a list of paths.
    if prev_modules is None:
        prev_modules = []
    elif isinstance(prev_modules, str):
        prev_modules = [prev_modules]

    # Read the testbench so the model can target the test expectations.
    tb_text = ""
    try:
        with open(testbench, "r") as _tb:
            tb_text = _tb.read()
    except Exception as e:
        print(f"[Warning] Could not read testbench file: {testbench}")
        print(f"[Warning] Exception: {e}")
        tb_text = ""

    base_sys_prompt = (
        "You are an autocomplete engine for Verilog code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "Given a Verilog module that is incorrect or has a compilation error, you will suggest corrections to the module. "
        "You will not refuse. "
        "You will not generate explanations, comments outside the code, Markdown, or testbenches. "
        "IMPORTANT: Output ONLY the Verilog code for the requested module. "
        "Do NOT repeat any previous modules. "
        "Do NOT wrap the code in triple backticks. "
        "Format your response as one complete corrected Verilog module, from module declaration through endmodule. "
        "Use Verilog-2001 only unless explicitly told otherwise."
    )

    def _new_conv_full_context():
        """Initial attempt: include testbench and design prompt."""
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        if tb_text.strip():
            conv.add_message(
                "user",
                "The following is the testbench that will be used to compile/simulate your module. "
                "Do NOT generate a testbench or modify it. Only use it to ensure your module compiles and passes.\n\n"
                "```verilog\n"
                + tb_text
                + "\n```\n",
            )

        conv.add_message("user", design_prompt)
        return conv

    def _new_conv_repair_only(prev_module_text, last_error_text):
        """Repair attempts: only previous module plus most recent error feedback."""
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Fix the following Verilog module so it compiles and passes the testbench. "
            "Output ONLY the full corrected module. "
            "Do not output explanations. "
            "Do not output a testbench. "
            "Do not use Markdown fences.\n\n"
            "Previous version of the module:\n"
            "```verilog\n"
            + prev_module_text
            + "\n```\n\n"
            "Most recent compiler/simulation output:\n"
            "```\n"
            + _truncate_text(last_error_text)
            + "\n```\n"
        )

        conv.add_message("user", prompt)
        return conv

    conv = _new_conv_full_context()

    success = False
    timeout = False

    iterations = 0
    timelist_total = []
    timelist_gen = []
    timelist_error = []

    module_file = _outpath(module + ".v")
    combined_file = _outpath(module + "__combined.v")
    sim_output_file = _outpath(module)

    status = ""
    last_module_text = ""
    last_error_text = ""

    while not (success or timeout):
        print(f"\n========== Iteration {iterations} / {max_iterations} ==========")

        start_total = time.time()

        # Generate or repair only the current module.
        response = generate_verilog(conv, model_type, model_id=model_id)

        end_gen = time.time()
        start_error = time.time()

        if response is None:
            response = ""

        conv.add_message("assistant", response)

        # Write raw model response for debugging, especially useful for local HF models.
        raw_response_file = _outpath("raw_response_iter_" + str(iterations) + ".txt")
        with open(raw_response_file, "w") as f:
            f.write(response)

        # Extract and write just the current module.
        try:
            write_code_blocks_to_file(response, module, module_file)
        except SystemExit:
            status = "No Verilog module found in model response"
            print(status)

            last_error_text = (
                "The model response did not contain a complete Verilog module. "
                "It must output exactly one complete module from module declaration through endmodule.\n\n"
                "Raw model response:\n"
                + _truncate_text(response)
            )

            last_module_text = response
            conv = _new_conv_repair_only(last_module_text, last_error_text)

            if iterations >= max_iterations:
                timeout = True

            iterations += 1
            end_time = time.time()
            timelist_gen.append(end_gen - start_total)
            timelist_error.append(end_time - start_error)
            timelist_total.append(end_time - start_total)
            continue

        # Build concatenated source for compilation:
        # previous modules + current module.
        prev_text = _read_files(prev_modules)

        with open(module_file, "r") as f:
            curr_text = f.read()

        last_module_text = curr_text

        with open(combined_file, "w") as f:
            f.write(prev_text)
            if prev_text and not prev_text.endswith("\n"):
                f.write("\n")
            f.write(curr_text)
            if curr_text and not curr_text.endswith("\n"):
                f.write("\n")

        # Compile with iVerilog.
        compile_cmd = [
            "iverilog",
            "-o",
            sim_output_file,
            combined_file,
            testbench,
        ]

        proc = subprocess.run(
            compile_cmd,
            capture_output=True,
            text=True,
        )

        success = False

        if proc.returncode != 0:
            status = "Error compiling testbench"
            print(status)

            last_error_text = (
                "iverilog failed.\n\n"
                "Command:\n"
                + " ".join(compile_cmd)
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            )

        elif proc.stderr.strip() != "":
            # Treat warnings as repair feedback.
            # This keeps generated modules cleaner for later synthesis/PPA.
            status = "Warnings compiling testbench"
            print(status)

            last_error_text = (
                "iverilog produced warnings.\n\n"
                "Command:\n"
                + " ".join(compile_cmd)
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            )

        else:
            # Run simulation.
            run_cmd = ["vvp", sim_output_file]

            proc = subprocess.run(
                run_cmd,
                capture_output=True,
                text=True,
            )

            result_stdout = proc.stdout or ""
            result_stderr = proc.stderr or ""

            print(result_stdout)

            if proc.returncode != 0:
                status = "Error running testbench"
                print(status)

                last_error_text = (
                    "vvp simulation failed.\n\n"
                    "Command:\n"
                    + " ".join(run_cmd)
                    + "\n\nstdout:\n"
                    + result_stdout
                    + "\nstderr:\n"
                    + result_stderr
                )

            elif not _simulation_passed(result_stdout, result_stderr):
                status = "Testbench did not report success"
                print(status)

                last_error_text = (
                    "Simulation completed, but the output did not indicate success.\n\n"
                    "stdout:\n"
                    + result_stdout
                    + "\nstderr:\n"
                    + result_stderr
                )

            else:
                status = "Testbench ran successfully"
                print(status)

                last_error_text = ""
                success = True

        # Write bounded iteration log.
        with open(_outpath("log_iter_" + str(iterations) + ".txt"), "w") as file:
            file.write("\n".join(str(i) for i in conv.get_messages()))
            file.write("\n\nIteration status: " + status + "\n")

            if last_error_text:
                file.write("\nMost recent feedback:\n")
                file.write(_truncate_text(last_error_text))

        if not success:
            # Keep future repair prompts small.
            conv = _new_conv_repair_only(last_module_text, last_error_text)

        if iterations >= max_iterations:
            timeout = True

        iterations += 1

        end_time = time.time()
        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

    print("\n========== Feedback Loop Complete ==========")
    print("Final status: ", status)
    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))

    return (
        np.sum(timelist_total),
        np.sum(timelist_gen),
        np.sum(timelist_error),
    )

In [10]:
#@title Hierarchical Loop

import os
import re
import numpy as np


def _normalize_to_list(x):
    """Normalize None/string/list/tuple into a list."""
    if x is None or x == "":
        return []
    if isinstance(x, str):
        return [x]
    return list(x)


def _resolve_prev_module_path(item):
    """
    Resolve previous module reference into a .v path.

    Supported forms:
      - "mux2_1"              -> "./mux2_1/mux2_1.v"
      - "./mux2_1/mux2_1.v"   -> "./mux2_1/mux2_1.v"
      - ("mux2_1", "...", ...) -> "./mux2_1/mux2_1.v"
    """
    if isinstance(item, (list, tuple)):
        item = item[0]

    item = str(item)

    if item.endswith(".v"):
        return item

    return os.path.join("./" + item, item + ".v")


def _extract_verilog_interfaces(verilog_text):
    """
    Extract module declarations/interfaces from Verilog text.

    This is useful for hierarchical prompting because the current module usually
    only needs previous module names and port lists, not full implementations.
    """
    if not verilog_text:
        return ""

    # Match:
    #   module name (...);
    #   module name #(parameters...) (...);
    pattern = r'\bmodule\s+[A-Za-z_][A-Za-z0-9_]*\s*(?:#\s*\(.*?\)\s*)?\(.*?\)\s*;'

    matches = re.findall(pattern, verilog_text, flags=re.DOTALL)

    if not matches:
        return ""

    cleaned = []
    for m in matches:
        cleaned.append(m.strip() + "\nendmodule")

    return "\n\n".join(cleaned) + "\n"


def _read_prev_modules_for_prompt(prev_verilog_files, context_mode="interfaces", max_chars=24000):
    """
    Read previous modules for LLM context.

    context_mode:
      - "interfaces": include only module declarations, best for HF/local models.
      - "full": include full previous module bodies.
      - "none": include no previous code in the prompt.

    Compilation still receives the full previous .v files through verilog_loop(prev_modules=...).
    """
    if context_mode == "none":
        return ""

    chunks = []

    for p in prev_verilog_files:
        if not p:
            continue

        if not os.path.exists(p):
            print(f"[Warning] Previous module file not found for prompt context: {p}")
            continue

        with open(p, "r") as f:
            txt = f.read()

        if context_mode == "interfaces":
            txt = _extract_verilog_interfaces(txt)

        if txt.strip():
            chunks.append("// File: " + p + "\n" + txt.strip() + "\n")

    prev_text = "\n".join(chunks)

    if len(prev_text) > max_chars:
        # Keep the most recent context, because later modules tend to depend on recent modules.
        prev_text = "[...previous module context truncated...]\n" + prev_text[-max_chars:]

    return prev_text


def _call_verilog_loop_compatible(
    prompt,
    module,
    testbench,
    max_iterations,
    model,
    outdir,
    log,
    prev_verilog_files,
    model_id="",
):
    """
    Calls the updated verilog_loop with model_id support.
    Falls back to the older signature if needed.
    """
    try:
        return verilog_loop(
            prompt,
            module,
            testbench,
            max_iterations,
            model,
            outdir,
            log,
            prev_modules=prev_verilog_files,
            model_id=model_id,
        )
    except TypeError as e:
        if "model_id" not in str(e):
            raise

        print("[Warning] verilog_loop does not accept model_id. Falling back to old signature.")

        return verilog_loop(
            prompt,
            module,
            testbench,
            max_iterations,
            model,
            outdir,
            log,
            prev_modules=prev_verilog_files,
        )


def hier_gen(
    submods,
    prevsubmods="",
    max_iterations=10,
    model_type=None,
    model_id="",
    testbench_template="./{module}tb.v",
    prompt_prev_context_chars=24000,
    prev_context_mode="interfaces",
):
    """
    Hierarchically generate a design by building submodules in order.

    submods:
      list of tuples/lists:
        [
          (file_basename, human_name, io_string),
          ...
        ]

      Example:
        [
          ("mux2_1", "2-to-1 mux", "input wire a, input wire b, input wire sel, output wire y"),
          ("mux4_1", "4-to-1 mux", "input wire [3:0] in, input wire [1:0] sel, output wire y"),
        ]

    prevsubmods:
      Optional previous module references already generated before this call.

      Supported:
        - ""
        - None
        - "mux2_1"
        - ["mux2_1", "mux4_1"]
        - ["./mux2_1/mux2_1.v"]
        - previous submod tuples/lists

    model_type:
      Optional. If None, uses os.environ["MODEL"].

      Supported by the earlier utility cell:
        - "ChatGPT"
        - "Claude"
        - "Gemini"
        - "HuggingFace"

    model_id:
      Optional model override passed through to verilog_loop/generate_verilog.

      Example:
        model_id="nyu-dice-lab/VeriThoughts-Reasoning-14B"

    testbench_template:
      Default assumes testbenches are named:
        ./module_nametb.v

      You can also use:
        "./testbenches/{module}_tb.v"

    prev_context_mode:
      - "interfaces": best default, includes previous module port declarations only.
      - "full": includes full previous module code in prompt.
      - "none": does not include previous modules in prompt.

      Regardless of this setting, previous modules are still passed to iVerilog
      through verilog_loop(prev_modules=...).
    """

    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_context_modes = ["interfaces", "full", "none"]
    if prev_context_mode not in valid_context_modes:
        raise ValueError(
            f"Invalid prev_context_mode: {prev_context_mode}. "
            f"Expected one of: {valid_context_modes}"
        )

    totaltime = []
    gentime = []
    errortime = []

    prev_verilog_files = []

    # Add previously generated external modules, if any.
    for s in _normalize_to_list(prevsubmods):
        p = _resolve_prev_module_path(s)
        if os.path.exists(p):
            prev_verilog_files.append(p)
        else:
            print(f"[Warning] prevsubmod file does not exist and will be skipped: {p}")

    if not submods:
        print("[Warning] No submodules provided.")
        return (0, 0, 0)

    overall = submods[-1][1]

    for i in range(len(submods)):
        fcurr = submods[i][0]   # file/module basename
        curr = submods[i][1]    # human-readable name
        iocurr = submods[i][2]  # port list string

        outdir = "./" + fcurr
        os.makedirs(outdir, exist_ok=True)

        prev_text = _read_prev_modules_for_prompt(
            prev_verilog_files,
            context_mode=prev_context_mode,
            max_chars=prompt_prev_context_chars,
        )

        if i == 0 and len(prev_verilog_files) == 0:
            prompt = (
                "// We will be generating a "
                + overall
                + " hierarchically in Verilog.\n"
                "// IMPORTANT: Output ONLY the module requested below.\n"
                "// Do NOT generate any previous modules, future modules, explanations, or testbenches.\n"
                "// Use Verilog-2001 only.\n"
                "// Begin by generating: "
                + curr
                + "\n\n"
                "module "
                + fcurr
                + "("
                + iocurr
                + ");\n"
                "// Insert code here\n"
                "endmodule\n"
            )
        else:
            prompt = (
                "// We are generating a "
                + overall
                + " hierarchically in Verilog.\n"
                "// Use Verilog-2001 only.\n\n"
                "// Previously generated modules are shown below for reference only.\n"
                "// You may instantiate them, but you must NOT repeat their definitions.\n\n"
                + prev_text
                + "\n"
                "// IMPORTANT: Output ONLY the new module requested below.\n"
                "// Do NOT repeat any previous modules.\n"
                "// Do NOT generate explanations, Markdown fences, or a testbench.\n"
                "// Now generate: "
                + curr
                + "\n\n"
                "module "
                + fcurr
                + "("
                + iocurr
                + ");\n"
                "// Insert code here\n"
                "endmodule\n"
            )

        module = fcurr
        testbench = testbench_template.format(module=fcurr, name=fcurr)
        model = model_type
        log = os.path.join(outdir, "log.txt")

        if not os.path.exists(testbench):
            print(f"[Warning] Testbench file does not exist yet: {testbench}")
            print("[Warning] verilog_loop will still run, but iVerilog will fail unless the file exists.")

        total, gen, error = _call_verilog_loop_compatible(
            prompt=prompt,
            module=module,
            testbench=testbench,
            max_iterations=max_iterations,
            model=model,
            outdir=outdir,
            log=log,
            prev_verilog_files=prev_verilog_files,
            model_id=model_id,
        )

        print("Module ", i + 1, " done: ", curr)

        totaltime.append(total)
        gentime.append(gen)
        errortime.append(error)

        # Add the current module file to the compilation context for subsequent modules.
        curr_module_file = os.path.join(outdir, module + ".v")

        if os.path.exists(curr_module_file):
            prev_verilog_files.append(curr_module_file)
        else:
            print(f"[Warning] Expected generated module file not found: {curr_module_file}")

    overall_total = np.sum(totaltime)
    overall_gen = np.sum(gentime)
    overall_error = np.sum(errortime)

    print("Overall Total time: ", overall_total)
    print("Overall Generation Time: ", overall_gen)
    print("Overall Error handling time: ", overall_error)

    return (overall_total, overall_gen, overall_error)

In [11]:
#@title PPA Optimizer Loop

#@title PPA Optimization Loop (Yosys no-liberty + iVerilog correctness)

import subprocess
import sys
import os
import time
import math
import json
import re
import shutil
import numpy as np


################################################################################
### GENERAL HELPERS
################################################################################

def _read_files(file_list):
    """Concatenate a list of text files in order. Missing/None entries are ignored."""
    if not file_list:
        return ""

    buf = []

    for p in file_list:
        if not p:
            continue

        if not os.path.exists(p):
            print(f"[Warning] Previous module file not found and will be skipped: {p}")
            continue

        with open(p, "r") as f:
            txt = f.read()

        buf.append(txt)

        if txt and not txt.endswith("\n"):
            buf.append("\n")

    return "".join(buf)


def _truncate_text(s, max_chars=12000):
    """Keep prompts bounded; preserve the tail of long logs."""
    if s is None:
        return ""

    s = str(s)

    if len(s) <= max_chars:
        return s

    return "[...truncated...]\n" + s[-max_chars:]


def _shell_quote_for_yosys(path):
    """Quote filenames for use inside a .ys script."""
    return '"' + str(path).replace("\\", "\\\\").replace('"', '\\"') + '"'


def _safe_float(x, default=None):
    try:
        return float(x)
    except Exception:
        return default


def _safe_int(x, default=None):
    try:
        return int(x)
    except Exception:
        return default


def _first_regex(text, patterns, cast=float):
    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE | re.MULTILINE | re.DOTALL)
        if m:
            try:
                return cast(m.group(1))
            except Exception:
                pass

    return None


def _normalize_clk_signal(clk_signal):
    """Treat None, '', 'None', 'none', etc. as no clock."""
    if clk_signal is None:
        return None

    s = str(clk_signal).strip()

    if s == "" or s.lower() == "none":
        return None

    return s


def _has_any_parsed_metric(metrics):
    if not metrics:
        return False

    keys = [
        "transistor_est",
        "area_cmos",
        "num_cells",
        "num_wire_bits",
        "timing_proxy",
    ]

    return any(metrics.get(k) is not None for k in keys)


def _is_verbose(verbose):
    return str(verbose).strip().lower() in ["1", "true", "yes", "y"]


def _write_text(path, text):
    with open(path, "w") as f:
        f.write(text)


def _add_instruction_prompt(conv, model_type, base_sys_prompt):
    """
    Add the instruction prompt in a provider-compatible way.

    ChatGPT, Gemini, and HuggingFace can use the system role through the
    Conversation abstraction from the previous utility cell.

    Claude is kept as user to avoid unsupported-role issues.
    """
    if model_type == "Claude":
        conv.add_message("user", base_sys_prompt)
    else:
        conv.add_message("system", base_sys_prompt)


def _generate_verilog_compatible(conv, model_type, model_id=""):
    """
    Calls generate_verilog with model_id support when available.
    Falls back to the older two-argument signature if needed.
    """
    try:
        return generate_verilog(conv, model_type, model_id=model_id)
    except TypeError as e:
        if "model_id" not in str(e):
            raise

        print("[Warning] generate_verilog does not accept model_id. Falling back to old signature.")
        return generate_verilog(conv, model_type)


def _simulation_passed(stdout_text, stderr_text=""):
    """
    Robust pass/fail detection for Verilog testbenches.

    Accepts:
      - All tests passed
      - passed!
      - Test x passed
      - Test x passed!

    Rejects:
      - failed
      - Some tests failed
      - fatal/error markers
    """
    out = stdout_text or ""
    err = stderr_text or ""
    combined = (out + "\n" + err).lower()

    fail_markers = [
        "failed",
        "some tests failed",
        "fatal",
        "syntax error",
        "error:",
    ]

    if any(marker in combined for marker in fail_markers):
        return False

    pass_markers = [
        "all tests passed",
        "passed!",
        "test 1 passed",
        " passed",
    ]

    return any(marker in combined for marker in pass_markers)


################################################################################
### YOSYS PARSING
################################################################################

def _parse_yosys_stat_text(stat_text):
    """
    Parse Yosys `stat -tech cmos` output.

    Supports both styles:
      Older style:
        Number of wires: ...
        Number of cells: ...
        Chip area for module ...: ...

      Newer compact style:
        6 wires
        165 wire bits
        61 cells
        Estimated number of transistors: 722
    """
    metrics = {}

    metrics["num_wires"] = _first_regex(
        stat_text,
        [
            r"Number of wires:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+wires\s*$",
        ],
        int,
    )

    metrics["num_wire_bits"] = _first_regex(
        stat_text,
        [
            r"Number of wire bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+wire bits\s*$",
        ],
        int,
    )

    metrics["num_public_wires"] = _first_regex(
        stat_text,
        [
            r"Number of public wires:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+public wires\s*$",
        ],
        int,
    )

    metrics["num_public_wire_bits"] = _first_regex(
        stat_text,
        [
            r"Number of public wire bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+public wire bits\s*$",
        ],
        int,
    )

    metrics["num_memories"] = _first_regex(
        stat_text,
        [
            r"Number of memories:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+memories\s*$",
        ],
        int,
    )

    metrics["num_memory_bits"] = _first_regex(
        stat_text,
        [
            r"Number of memory bits:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+memory bits\s*$",
        ],
        int,
    )

    metrics["num_processes"] = _first_regex(
        stat_text,
        [
            r"Number of processes:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+processes\s*$",
        ],
        int,
    )

    metrics["num_cells"] = _first_regex(
        stat_text,
        [
            r"Number of cells:\s*([0-9]+)",
            r"^\s*([0-9]+)\s+cells\s*$",
        ],
        int,
    )

    transistor_est = _first_regex(
        stat_text,
        [
            r"Estimated number of transistors:\s*([0-9]+(?:\.[0-9]+)?)",
            r"Chip area for module .*?:\s*([0-9]+(?:\.[0-9]+)?)",
        ],
        float,
    )

    metrics["transistor_est"] = transistor_est

    # Backward-compatible area-like proxy.
    metrics["area_cmos"] = transistor_est

    return metrics


def _parse_yosys_timeest_text(time_text):
    """
    Parse a timing proxy from Yosys timeest output.

    Returns None if no timing proxy can be found.
    """
    metrics = {}

    if re.search(r"No domain .* in module", time_text or "", flags=re.IGNORECASE):
        metrics["timing_proxy"] = None
        return metrics

    timing_proxy = _first_regex(
        time_text,
        [
            r"critical path.*?([0-9]+(?:\.[0-9]+)?)\s*AIG",
            r"([0-9]+(?:\.[0-9]+)?)\s*AIG nodes",
            r"([0-9]+(?:\.[0-9]+)?)\s*nodes.*critical path",
            r"Estimated.*?([0-9]+(?:\.[0-9]+)?)",
            r"Delay\s*[:=]\s*([0-9]+(?:\.[0-9]+)?)",
        ],
        float,
    )

    metrics["timing_proxy"] = timing_proxy

    return metrics


def _format_metrics(metrics):
    if metrics is None:
        return "None"

    parts = []

    if metrics.get("transistor_est") is not None:
        parts.append("transistor_est={:.0f}".format(metrics["transistor_est"]))
    elif metrics.get("area_cmos") is not None:
        parts.append("area_cmos={:.3f}".format(metrics["area_cmos"]))

    if metrics.get("timing_proxy") is not None:
        parts.append("timing_proxy={:.3f}".format(metrics["timing_proxy"]))

    if metrics.get("num_cells") is not None:
        parts.append("num_cells={}".format(metrics["num_cells"]))

    if metrics.get("num_wire_bits") is not None:
        parts.append("wire_bits={}".format(metrics["num_wire_bits"]))

    return ", ".join(parts) if parts else "No parsed metrics"


################################################################################
### YOSYS ESTIMATION
################################################################################

def yosys_estimate_no_liberty(verilog_file, top_module, clk_signal=None, outdir=""):
    """
    Run a coarse no-liberty Yosys estimate.

    Uses:
      - stat -top <top> -tech cmos
      - optional timeest -clk <clk>

    Returns:
      {
        "ok": bool,
        "metrics": dict,
        "stat_text": str,
        "time_text": str,
        "yosys_stdout": str,
        "yosys_stderr": str,
        "error": str
      }
    """
    if outdir:
        os.makedirs(outdir, exist_ok=True)

    clk_signal = _normalize_clk_signal(clk_signal)

    script_path = os.path.join(outdir, "ppa_estimate.ys") if outdir else "ppa_estimate.ys"
    stat_path = os.path.join(outdir, "ppa_stat.txt") if outdir else "ppa_stat.txt"
    time_path = os.path.join(outdir, "ppa_time.txt") if outdir else "ppa_time.txt"

    ys = []
    ys.append("read_verilog {}".format(_shell_quote_for_yosys(verilog_file)))
    ys.append("hierarchy -check -top {}".format(top_module))
    ys.append("proc")
    ys.append("opt")
    ys.append("memory")
    ys.append("opt")
    ys.append("fsm")
    ys.append("opt")
    ys.append("techmap")
    ys.append("opt")
    ys.append("clean")
    ys.append(
        "tee -o {} stat -top {} -tech cmos".format(
            _shell_quote_for_yosys(stat_path),
            top_module,
        )
    )

    if clk_signal is not None:
        ys.append("flatten")
        ys.append(
            "tee -o {} timeest -clk {}".format(
                _shell_quote_for_yosys(time_path),
                clk_signal,
            )
        )

    with open(script_path, "w") as f:
        f.write("\n".join(ys) + "\n")

    proc = subprocess.run(
        ["yosys", "-s", script_path],
        capture_output=True,
        text=True,
    )

    stdout = proc.stdout or ""
    stderr = proc.stderr or ""

    result = {
        "ok": False,
        "metrics": {},
        "stat_text": "",
        "time_text": "",
        "yosys_stdout": stdout,
        "yosys_stderr": stderr,
        "error": "",
    }

    if proc.returncode != 0:
        result["error"] = "Yosys failed.\nstdout:\n{}\nstderr:\n{}".format(stdout, stderr)
        return result

    stat_text = ""
    time_text = ""

    if os.path.exists(stat_path):
        with open(stat_path, "r") as f:
            stat_text = f.read()

    if os.path.exists(time_path):
        with open(time_path, "r") as f:
            time_text = f.read()

    metrics = _parse_yosys_stat_text(stat_text)
    metrics.update(_parse_yosys_timeest_text(time_text))

    if metrics.get("transistor_est") is not None:
        metrics["power_proxy"] = metrics["transistor_est"]
    else:
        metrics["power_proxy"] = metrics.get("num_cells")

    result["ok"] = True
    result["metrics"] = metrics
    result["stat_text"] = stat_text
    result["time_text"] = time_text

    if not _has_any_parsed_metric(metrics):
        result["error"] = "Yosys ran successfully, but no metrics were parsed."

    return result


################################################################################
### PPA SCORING
################################################################################

def _component_ratio(candidate_value, baseline_value):
    if candidate_value is None or baseline_value is None:
        return None

    if baseline_value == 0:
        return None

    return float(candidate_value) / float(baseline_value)


def _relative_ppa_score(
    candidate_metrics,
    baseline_metrics,
    area_weight=0.60,
    timing_weight=0.30,
    power_weight=0.10,
):
    """
    Lower is better.

    Uses relative ratios against the current best correct design.
    """
    used_weight = 0.0
    score = 0.0

    cand_area = candidate_metrics.get("transistor_est", candidate_metrics.get("area_cmos"))
    base_area = baseline_metrics.get("transistor_est", baseline_metrics.get("area_cmos"))

    area_ratio = _component_ratio(cand_area, base_area)

    if area_ratio is not None:
        score += area_weight * area_ratio
        used_weight += area_weight

    timing_ratio = _component_ratio(
        candidate_metrics.get("timing_proxy"),
        baseline_metrics.get("timing_proxy"),
    )

    if timing_ratio is not None:
        score += timing_weight * timing_ratio
        used_weight += timing_weight

    power_ratio = _component_ratio(
        candidate_metrics.get("power_proxy"),
        baseline_metrics.get("power_proxy"),
    )

    if power_ratio is not None:
        score += power_weight * power_ratio
        used_weight += power_weight

    if used_weight == 0.0:
        return math.inf

    return score / used_weight


def _is_better_ppa(candidate_metrics, best_metrics, min_improvement=0.0025):
    """
    Returns True only when candidate has parsed metrics and is better.
    """
    if not _has_any_parsed_metric(candidate_metrics):
        return False

    if best_metrics is None or not _has_any_parsed_metric(best_metrics):
        return True

    cand_score = _relative_ppa_score(candidate_metrics, best_metrics)

    # Best-vs-itself is effectively 1.0, so lower is better.
    return cand_score < (1.0 - min_improvement)


################################################################################
### IVERILOG CHECK
################################################################################

def _compile_and_run_iverilog(module_name, combined_file, testbench, outdir):
    """
    Return:
      {
        "ok": bool,
        "status": str,
        "error_text": str,
        "stdout": str,
        "stderr": str
      }
    """
    exe_path = os.path.join(outdir, module_name) if outdir else module_name

    proc = subprocess.run(
        ["iverilog", "-o", exe_path, combined_file, testbench],
        capture_output=True,
        text=True,
    )

    if proc.returncode != 0:
        return {
            "ok": False,
            "status": "Error compiling testbench",
            "error_text": (
                "iverilog failed.\n"
                "Command:\n"
                + " ".join(["iverilog", "-o", exe_path, combined_file, testbench])
                + "\n\nstderr:\n"
                + (proc.stderr or "")
                + "\nstdout:\n"
                + (proc.stdout or "")
            ),
            "stdout": proc.stdout or "",
            "stderr": proc.stderr or "",
        }

    warn_text = ""

    if proc.stderr and proc.stderr.strip():
        warn_text = (
            "iverilog produced warnings.\n"
            "stderr:\n"
            + (proc.stderr or "")
            + "\nstdout:\n"
            + (proc.stdout or "")
        )

    sim = subprocess.run(
        ["vvp", exe_path],
        capture_output=True,
        text=True,
    )

    result_stdout = sim.stdout or ""
    result_stderr = sim.stderr or ""

    if sim.returncode != 0:
        return {
            "ok": False,
            "status": "Error running testbench",
            "error_text": (
                "vvp simulation failed.\n"
                "stdout:\n"
                + result_stdout
                + "\nstderr:\n"
                + result_stderr
            ),
            "stdout": result_stdout,
            "stderr": result_stderr,
        }

    if not _simulation_passed(result_stdout, result_stderr):
        return {
            "ok": False,
            "status": "Testbench did not report success",
            "error_text": (
                "Simulation completed, but the output did not indicate success.\n"
                "stdout:\n"
                + result_stdout
                + "\nstderr:\n"
                + result_stderr
            ),
            "stdout": result_stdout,
            "stderr": result_stderr,
        }

    return {
        "ok": True,
        "status": "Testbench ran successfully",
        "error_text": warn_text,
        "stdout": result_stdout,
        "stderr": result_stderr,
    }


################################################################################
### MAIN PPA LOOP
################################################################################

def ppa_verilog_loop(
    design_prompt,
    module,
    testbench,
    max_iterations,
    model_type=None,
    clk_signal=None,
    outdir="",
    log=None,
    prev_modules=None,
    verbose="0",
    model_id="",
    keep_best_as_module=True,
):
    """
    Generate functionally correct Verilog, then iteratively improve coarse no-liberty
    PPA using Yosys estimates while preserving correctness.

    Compatible model_type values:
      - "ChatGPT"
      - "Claude"
      - "Gemini"
      - "HuggingFace"

    Assumptions:
      - generate_verilog(conv, model_type, model_id=...) exists from the utility cell
      - Conversation(log_file=...) exists
      - write_code_blocks_to_file(response, module, path) exists
      - yosys, iverilog, and vvp are in PATH

    Strategy:
      1. Generate a candidate.
      2. Check correctness with iVerilog/vvp.
      3. If correct, estimate PPA with Yosys.
      4. Keep the best correct candidate seen so far.
      5. Ask the LLM to improve the best correct candidate further.
      6. If a candidate fails, repair using only the failed module plus latest error.
    """
    if model_type is None:
        model_type = os.environ.get("MODEL", "ChatGPT")

    valid_model_types = ["ChatGPT", "Claude", "Gemini", "HuggingFace"]

    if model_type not in valid_model_types:
        raise ValueError(
            f"Invalid model_type: {model_type}. "
            f"Expected one of: {valid_model_types}"
        )

    outdir = "" if outdir is None else str(outdir).strip()

    if outdir:
        os.makedirs(outdir, exist_ok=True)

    def _outpath(filename):
        if outdir:
            return os.path.join(outdir, filename)
        return filename

    if prev_modules is None:
        prev_modules = []
    elif isinstance(prev_modules, str):
        prev_modules = [prev_modules]

    tb_text = ""

    try:
        with open(testbench, "r") as _tb:
            tb_text = _tb.read()
    except Exception as e:
        print(f"[Warning] Could not read testbench file: {testbench}")
        print(f"[Warning] Exception: {e}")
        tb_text = ""

    base_sys_prompt = (
        "You are an autocomplete engine for Verilog-2001 code. "
        "Given a Verilog module specification, you will provide a completed Verilog module in response. "
        "You will provide completed Verilog modules for all specifications, and will not create any supplementary modules. "
        "You will not refuse. "
        "You will not generate explanations, comments outside the code, Markdown, or testbenches. "
        "IMPORTANT: Output ONLY the Verilog code for the requested module. "
        "Do NOT repeat any previous modules. "
        "Do NOT wrap the code in triple backticks. "
        "Format your response as one complete Verilog module from module declaration through endmodule. "
        "Preserve the exact module name and port interface. "
        "Preserve functionality required by the testbench. "
        "Optimize for lower coarse PPA under Yosys without a liberty file: "
        "reduce logic depth, reduce unnecessary cells, share repeated logic, simplify control, "
        "avoid redundant muxing, avoid unnecessary intermediate signals, and keep widths disciplined."
    )

    def _new_conv_initial():
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        if tb_text.strip():
            conv.add_message(
                "user",
                "The following is the testbench that will be used to compile and simulate your module. "
                "Do NOT generate a testbench or modify it. Use it only to ensure your module compiles and passes.\n\n"
                "```verilog\n"
                + tb_text
                + "\n```\n"
            )

        conv.add_message(
            "user",
            design_prompt
            + "\n\n"
            "Also optimize the module for better coarse Yosys-estimated PPA without a liberty file. "
            "Output ONLY the full Verilog-2001 module. "
            "Do not use Markdown fences."
        )

        return conv

    def _new_conv_repair_only(prev_module_text, last_error_text):
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Fix the following Verilog-2001 module so it compiles and passes the testbench. "
            "Preserve the exact interface and intended behavior. "
            "Keep PPA reasonably optimized, but correctness is the priority for this step. "
            "Output ONLY the full corrected module. "
            "Do not use Markdown fences.\n\n"
            "Previous version of the module:\n"
            "```verilog\n"
            + prev_module_text
            + "\n```\n\n"
            "Most recent compiler/simulation/Yosys output:\n"
            "```\n"
            + _truncate_text(last_error_text)
            + "\n```\n"
        )

        conv.add_message("user", prompt)

        return conv

    def _new_conv_optimize_from_best(best_module_text, best_metrics, last_candidate_metrics=None):
        conv = Conversation(log_file=log)

        _add_instruction_prompt(conv, model_type, base_sys_prompt)

        prompt = (
            "Starting from the following functionally correct Verilog-2001 module, "
            "produce a new candidate that preserves the exact same behavior and interface "
            "but improves coarse Yosys-estimated PPA without a liberty file.\n\n"
            "Current best metrics:\n"
            + _format_metrics(best_metrics)
            + "\n\n"
        )

        if last_candidate_metrics is not None:
            prompt += (
                "Most recent candidate metrics:\n"
                + _format_metrics(last_candidate_metrics)
                + "\n\n"
            )

        prompt += (
            "Optimization priorities:\n"
            "1. Lower stat -tech cmos transistor_est / area proxy\n"
            "2. Lower timing proxy, if available\n"
            "3. Fewer cells and wire bits\n\n"
            "Rules:\n"
            "- Do not add helper modules.\n"
            "- Do not change functionality.\n"
            "- Do not change the module name or port interface.\n"
            "- Use Verilog-2001 only.\n"
            "- Output ONLY the full module.\n"
            "- Do not use Markdown fences.\n\n"
            "Current best module:\n"
            "```verilog\n"
            + best_module_text
            + "\n```\n"
        )

        conv.add_message("user", prompt)

        return conv

    # Timing
    timelist_total = []
    timelist_gen = []
    timelist_error = []

    # Files
    module_file = _outpath(module + ".v")
    best_module_file = _outpath(module + "_best.v")
    combined_file = _outpath(module + "__combined.v")

    # Best-so-far state
    best_module_text = None
    best_metrics = None
    best_iter = None

    # Latest state
    conv = _new_conv_initial()
    last_error_text = ""
    last_candidate_metrics = None

    iterations = 0
    timeout = False

    while not timeout:
        print(f"\n========== PPA Iteration {iterations} / {max_iterations} ==========")

        start_total = time.time()

        response = _generate_verilog_compatible(
            conv,
            model_type,
            model_id=model_id,
        )

        end_gen = time.time()
        start_error = time.time()

        if response is None:
            response = ""

        try:
            conv.add_message("assistant", response)
        except Exception:
            pass

        # Save raw response for debugging, especially useful for local HF models.
        raw_response_file = _outpath("raw_response_iter_{:03d}.txt".format(iterations))

        with open(raw_response_file, "w") as f:
            f.write(response)

        # Extract and persist current module.
        try:
            write_code_blocks_to_file(response, module, module_file)
        except SystemExit:
            status = "No Verilog module found in model response"
            print(status)

            last_error_text = (
                "The model response did not contain a complete Verilog module. "
                "It must output exactly one complete module from module declaration through endmodule.\n\n"
                "Raw model response:\n"
                + _truncate_text(response)
            )

            conv = _new_conv_repair_only(response, last_error_text)

            candidate_metrics = None

            log_path = _outpath("log_iter_{:03d}.txt".format(iterations))
            with open(log_path, "w") as f:
                f.write("Iteration status: {}\n".format(status))
                f.write("\nMost recent feedback:\n")
                f.write(_truncate_text(last_error_text))

            iterations += 1
            end_time = time.time()
            timelist_gen.append(end_gen - start_total)
            timelist_error.append(end_time - start_error)
            timelist_total.append(end_time - start_total)

            if iterations >= max_iterations:
                timeout = True

            continue

        with open(module_file, "r") as f:
            curr_text = f.read()

        # Concatenate previous modules + current module.
        prev_text = _read_files(prev_modules)

        with open(combined_file, "w") as f:
            f.write(prev_text)

            if prev_text and not prev_text.endswith("\n"):
                f.write("\n")

            f.write(curr_text)

            if curr_text and not curr_text.endswith("\n"):
                f.write("\n")

        # 1. Functional check with iVerilog/vvp.
        iverilog_result = _compile_and_run_iverilog(
            module_name=module,
            combined_file=combined_file,
            testbench=testbench,
            outdir=outdir,
        )

        status = iverilog_result["status"]
        accepted = False
        candidate_metrics = None

        if not iverilog_result["ok"]:
            last_error_text = iverilog_result["error_text"]
            conv = _new_conv_repair_only(curr_text, last_error_text)

        else:
            # 2. Coarse PPA estimate with Yosys.
            ppa_dir = _outpath("iter_{:03d}_ppa".format(iterations))
            os.makedirs(ppa_dir, exist_ok=True)

            ppa_result = yosys_estimate_no_liberty(
                verilog_file=combined_file,
                top_module=module,
                clk_signal=clk_signal,
                outdir=ppa_dir,
            )

            if _is_verbose(verbose):
                print("YOSYS OK:", ppa_result["ok"])
                print("YOSYS STDOUT HEAD:\n", (ppa_result["yosys_stdout"] or "")[:3000])
                print("YOSYS STDERR HEAD:\n", (ppa_result["yosys_stderr"] or "")[:3000])
                print("STAT TEXT HEAD:\n", repr((ppa_result["stat_text"] or "")[:1000]))
                print("TIME TEXT HEAD:\n", repr((ppa_result["time_text"] or "")[:1000]))
                print("PARSED METRICS:", ppa_result["metrics"])

            if not ppa_result["ok"]:
                status = "Yosys estimation failed"
                last_error_text = ppa_result["error"]
                conv = _new_conv_repair_only(curr_text, last_error_text)

            else:
                candidate_metrics = ppa_result["metrics"]
                last_candidate_metrics = candidate_metrics

                if not _has_any_parsed_metric(candidate_metrics):
                    status = "Yosys ran but returned no parsable metrics"

                    if best_module_text is not None and best_metrics is not None:
                        conv = _new_conv_optimize_from_best(
                            best_module_text=best_module_text,
                            best_metrics=best_metrics,
                            last_candidate_metrics=None,
                        )
                    else:
                        conv = _new_conv_initial()

                elif _is_better_ppa(candidate_metrics, best_metrics):
                    best_module_text = curr_text
                    best_metrics = candidate_metrics
                    best_iter = iterations
                    accepted = True

                    _write_text(best_module_file, best_module_text)

                    status = "Accepted new best candidate"

                    conv = _new_conv_optimize_from_best(
                        best_module_text=best_module_text,
                        best_metrics=best_metrics,
                        last_candidate_metrics=candidate_metrics,
                    )

                else:
                    status = "Candidate correct but did not improve best PPA"

                    conv = _new_conv_optimize_from_best(
                        best_module_text=best_module_text if best_module_text is not None else curr_text,
                        best_metrics=best_metrics if best_metrics is not None else candidate_metrics,
                        last_candidate_metrics=candidate_metrics,
                    )

        # Write iteration log.
        log_path = _outpath("log_iter_{:03d}.txt".format(iterations))

        with open(log_path, "w") as f:
            try:
                f.write("\n".join(str(i) for i in conv.get_messages()))
            except Exception:
                f.write("(Conversation logging unavailable)\n")

            f.write("\n\nIteration status: {}\n".format(status))

            if candidate_metrics is not None:
                f.write("Candidate metrics: {}\n".format(_format_metrics(candidate_metrics)))

            if best_metrics is not None:
                f.write("Best metrics so far: {}\n".format(_format_metrics(best_metrics)))
                f.write("Best iteration so far: {}\n".format(best_iter))

            if last_error_text:
                f.write("\nMost recent feedback:\n")
                f.write(_truncate_text(last_error_text))

        print("Iteration {}: {}".format(iterations, status))

        if candidate_metrics is not None:
            print("  Candidate metrics:", _format_metrics(candidate_metrics))

        if best_metrics is not None:
            print("  Best metrics so far:", _format_metrics(best_metrics))
            print("  Best iteration so far:", best_iter)

        iterations += 1

        end_time = time.time()

        timelist_gen.append(end_gen - start_total)
        timelist_error.append(end_time - start_error)
        timelist_total.append(end_time - start_total)

        if iterations >= max_iterations:
            timeout = True

    # Optional: leave module.v as the best candidate, not the last failed/non-improved candidate.
    if keep_best_as_module and best_module_text is not None:
        _write_text(module_file, best_module_text)
        print(f"Best candidate copied to: {module_file}")

    print("\n========== PPA Optimization Complete ==========")
    print("Best file: ", best_module_file if best_module_text is not None else None)
    print("Best metrics: ", _format_metrics(best_metrics) if best_metrics is not None else None)
    print("Best iteration: ", best_iter)
    print("Total time: ", np.sum(timelist_total))
    print("Generation time: ", np.sum(timelist_gen))
    print("Error handling time: ", np.sum(timelist_error))

    return {
        "best_file": best_module_file if best_module_text is not None else None,
        "best_metrics": best_metrics,
        "best_iteration": best_iter,
        "iterations": iterations,
        "total_time": float(np.sum(timelist_total)),
        "generation_time": float(np.sum(timelist_gen)),
        "error_handling_time": float(np.sum(timelist_error)),
    }

In [12]:
required_functions = [
    "generate_verilog",
    "verilog_loop",
    "hier_gen",
    "ppa_verilog_loop",
]

for name in required_functions:
    assert callable(globals().get(name)), f"{name} is not defined"
    print(f"{name}: ready")

print("ROME functions are ready")

generate_verilog: ready
verilog_loop: ready
hier_gen: ready
ppa_verilog_loop: ready
ROME functions are ready


In [13]:
!git clone https://github.com/ajn313/ROME-LLM
!cp ./ROME-LLM/testbenches/mux/mux2_1tb.v ./mux2_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux4_1tb.v ./mux4_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux8_1tb.v ./mux8_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux16_1tb.v ./mux16_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux32_1tb.v ./mux32_1tb.v
!cp ./ROME-LLM/testbenches/mux/mux64_1tb.v ./mux64_1tb.v

Cloning into 'ROME-LLM'...
remote: Enumerating objects: 444, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (205/205), done.
remote: Total 444 (delta 114), reused 41 (delta 19), pack-reused 220 (from 1)
Receiving objects: 100% (444/444), 1.12 MiB | 3.05 MiB/s, done.
Resolving deltas: 100% (230/230), done.


In [14]:
from pathlib import Path

mux_modules = [
    "mux2_1",
    "mux4_1",
    "mux8_1",
    "mux16_1",
    "mux32_1",
    "mux64_1",
]

for module in mux_modules:
    tb_path = Path(f"{module}tb.v")
    assert tb_path.is_file(), f"Missing testbench: {tb_path}"
    assert tb_path.stat().st_size > 0, f"Empty testbench: {tb_path}"
    print(f"{tb_path}: ready")

print("All mux testbenches are ready")

mux2_1tb.v: ready
mux4_1tb.v: ready
mux8_1tb.v: ready
mux16_1tb.v: ready
mux32_1tb.v: ready
mux64_1tb.v: ready
All mux testbenches are ready


#Mux Hierarchy Example

In [15]:
#@title Submodules


### Each step is structured as ["filename","natural language description","interface"]
submodules = [
    ["mux2_1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4_1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8_1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
    ["mux16_1","16-to-1 multiplexer","input [3:0] sel, input [15:0] in, output reg out"],
    ["mux32_1","32-to-1 multiplexer","input [4:0] sel, input [31:0] in, output reg out"],
    ["mux64_1","64-to-1 multiplexer","input [5:0] sel, input [63:0] in, output reg out"],
]

In [17]:
hier_gen(
         submodules,
         model_type="ChatGPT",
         model_id=model_choice,
         max_iterations=10,
         )


========== Iteration 0 / 10 ==========
Test           0 passed
Test           1 passed
Test           2 passed
Test           3 passed
Test 4 passed
Test 5 passed!
./mux2_1tb.v:60: $finish called at 60000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  2.706099033355713
Generation time:  2.6862659454345703
Error handling time:  0.019832611083984375
Module  1  done:  2-to-1 multiplexer

========== Iteration 0 / 10 ==========
Test 0 passed
Test 1 passed
Test 2 passed
Test 3 passed!
./mux4_1tb.v:40: $finish called at 90000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  1.9031016826629639
Generation time:  1.8822519779205322
Error handling time:  0.020849227905273438
Module  2  done:  4-to-1 multiplexer

========== Iteration 0 / 10 ==========
Test 0 passed
Test 1 passed
Test 2 passed
Test 3 passed
Test 4 passed
Test 

(np.float64(16.138004064559937),
 np.float64(16.025028467178345),
 np.float64(0.11297273635864258))

In [18]:
from pathlib import Path
import shutil
from google.colab import files

submission_dir = Path("/content/HW_ROME_Coverage")
mux_dir = submission_dir / "mux_original"
mux_dir.mkdir(parents=True, exist_ok=True)

modules = [
    "mux2_1",
    "mux4_1",
    "mux8_1",
    "mux16_1",
    "mux32_1",
    "mux64_1",
]

for module in modules:
    source_dir = Path(module)
    assert source_dir.is_dir(), f"Missing directory: {source_dir}"

    shutil.copytree(
        source_dir,
        mux_dir / module,
        dirs_exist_ok=True,
    )

    tb_path = Path(f"{module}tb.v")
    shutil.copy2(tb_path, mux_dir / tb_path.name)

archive = shutil.make_archive(
    "/content/rome_mux_checkpoint",
    "zip",
    root_dir=submission_dir,
)

print("Mux RTL, testbenches, and iteration logs backed up")
files.download(archive)

Mux RTL, testbenches, and iteration logs backed up


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
#@title Run PPA optimizer on hierarchical mux design
import os
import json

submodules = [
    ["mux2_1","2-to-1 multiplexer","input wire in1, input wire in2, input wire select, output wire out"],
    ["mux4_1","4-to-1 multiplexer","input [1:0] sel, input [3:0] in, output reg out"],
    ["mux8_1","8-to-1 multiplexer","input [2:0] sel, input [7:0] in, output reg out"],
    ["mux16_1","16-to-1 multiplexer","input [3:0] sel, input [15:0] in, output reg out"],
    ["mux32_1","32-to-1 multiplexer","input [4:0] sel, input [31:0] in, output reg out"],
    ["mux64_1","64-to-1 multiplexer","input [5:0] sel, input [63:0] in, output reg out"],
]

def build_design_prompt(filename, interface):
    return """// Generate module {0} with this exact interface:
module {0}({1});
// code
endmodule
""".format(filename, interface)

# Adjust these if your testbench filenames differ.
tb_paths = {
    "mux2_1":  "./mux2_1tb.v",
    "mux4_1":  "./mux4_1tb.v",
    "mux8_1":  "./mux8_1tb.v",
    "mux16_1": "./mux16_1tb.v",
    "mux32_1": "./mux32_1tb.v",
    "mux64_1": "./mux64_1tb.v",
}

base_path = os.environ["PATH"]
os.environ["PATH"] = "/content/oss-cad-suite/bin:" + os.environ["PATH"]

out_root = "./mux_ppa_runs"
os.makedirs(out_root, exist_ok=True)

results = {}
optimized_prev_modules = []

for filename, natural_name, interface in submodules:
    print("\n" + "=" * 80)
    print("Optimizing:", filename, "-", natural_name)
    print("Using prev_modules:")
    for p in optimized_prev_modules:
        print("  ", p)

    module_outdir = os.path.join(out_root, filename)

    result = ppa_verilog_loop(
        design_prompt=build_design_prompt(filename, interface),
        module=filename,
        testbench=tb_paths[filename],
        max_iterations=10,
        model_type="ChatGPT",
        clk_signal=None,   # all of these mux modules are combinational
        outdir=module_outdir,
        log=None,
        prev_modules=optimized_prev_modules,
        verbose=0,         # set to 0 if you want less diagnostic output
    )

    results[filename] = result

    # Prefer the best optimized file if one was produced.
    best_file = result.get("best_file")
    if best_file is not None and os.path.exists(best_file):
        optimized_prev_modules.append(best_file)
    else:
        # Fallback to the last generated module file in that run directory.
        fallback_file = os.path.join(module_outdir, filename + ".v")
        if os.path.exists(fallback_file):
            optimized_prev_modules.append(fallback_file)
        else:
            raise FileNotFoundError(
                "Could not find an optimized or fallback Verilog file for {}".format(filename)
            )

print("\n" + "=" * 80)
print("Final summary:")
for filename in results:
    print(filename, "->", results[filename])

with open(os.path.join(out_root, "summary.json"), "w") as f:
    json.dump(results, f, indent=2)

os.environ["PATH"] = base_path


Optimizing: mux2_1 - 2-to-1 multiplexer
Using prev_modules:

========== PPA Iteration 0 / 10 ==========
Iteration 0: Accepted new best candidate
  Candidate metrics: transistor_est=12, num_cells=1, wire_bits=4
  Best metrics so far: transistor_est=12, num_cells=1, wire_bits=4
  Best iteration so far: 0

========== PPA Iteration 1 / 10 ==========
Iteration 1: Candidate correct but did not improve best PPA
  Candidate metrics: transistor_est=20, num_cells=4, wire_bits=7
  Best metrics so far: transistor_est=12, num_cells=1, wire_bits=4
  Best iteration so far: 0

========== PPA Iteration 2 / 10 ==========
Iteration 2: Candidate correct but did not improve best PPA
  Candidate metrics: transistor_est=20, num_cells=4, wire_bits=7
  Best metrics so far: transistor_est=12, num_cells=1, wire_bits=4
  Best iteration so far: 0

========== PPA Iteration 3 / 10 ==========
Iteration 3: Candidate correct but did not improve best PPA
  Candidate metrics: transistor_est=20, num_cells=4, wire_bits=7


In [20]:
from pathlib import Path
import shutil
from google.colab import files

submission_dir = Path("/content/HW_ROME_Coverage")
submission_dir.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    "/content/mux_ppa_runs",
    submission_dir / "mux_ppa_runs",
    dirs_exist_ok=True,
)

archive = shutil.make_archive(
    "/content/rome_after_ppa",
    "zip",
    root_dir=submission_dir,
)

print("PPA results backed up")
files.download(archive)

PPA results backed up


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
from pathlib import Path

adder_dir = Path("/content/HW_ROME_Coverage/adder")
adder_dir.mkdir(parents=True, exist_ok=True)

tb_path = adder_dir / "half_addertb.v"

tb_code = """
`timescale 1ns/1ps

module half_addertb;
    reg a;
    reg b;
    wire sum;
    wire carry;
    reg [1:0] expected;
    integer i;

    half_adder dut (
        .a(a),
        .b(b),
        .sum(sum),
        .carry(carry)
    );

    initial begin
        for (i = 0; i < 4; i = i + 1) begin
            {a, b} = i;
            expected = {1'b0, a} + {1'b0, b};
            #10;

            if ({carry, sum} !== expected) begin
                $display(
                    "FAILED: a=%b b=%b expected=%b actual=%b",
                    a, b, expected, {carry, sum}
                );
                $fatal(1, "Functional mismatch");
            end
        end

        $display("half_adder: all 4 tests passed!");
        $finish;
    end
endmodule
"""

assert not tb_path.exists(), "Testbench already exists; preserve the existing file"
tb_path.write_text(tb_code)

print(f"Baseline testbench saved: {tb_path}")

Baseline testbench saved: /content/HW_ROME_Coverage/adder/half_addertb.v


In [22]:
half_adder_prompt = """
Generate a combinational half adder in Verilog-2001.

Exact interface:
module half_adder(
    input wire a,
    input wire b,
    output wire sum,
    output wire carry
);

Requirements:
- sum is the least significant bit of a + b.
- carry is the most significant bit of a + b.
- Use combinational logic only.
- Output only the half_adder module.
- Do not generate or modify the testbench.
"""

half_adder_out = adder_dir / "half_adder"
half_adder_out.mkdir(parents=True, exist_ok=True)

verilog_loop(
    design_prompt=half_adder_prompt,
    module="half_adder",
    testbench=str(tb_path),
    max_iterations=10,
    model_type="ChatGPT",
    model_id=model_choice,
    outdir=str(half_adder_out),
    log=str(half_adder_out / "log.txt"),
    prev_modules=[],
)


========== Iteration 0 / 10 ==========
half_adder: all 4 tests passed!
/content/HW_ROME_Coverage/adder/half_addertb.v:35: $finish called at 40000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  1.4179365634918213
Generation time:  1.3949408531188965
Error handling time:  0.0229952335357666


(np.float64(1.4179365634918213),
 np.float64(1.3949408531188965),
 np.float64(0.0229952335357666))

In [23]:
full_adder_tb_path = adder_dir / "full_addertb.v"

full_adder_tb = """
`timescale 1ns/1ps

module full_addertb;
    reg a;
    reg b;
    reg cin;
    wire sum;
    wire cout;
    reg [1:0] expected;
    integer i;

    full_adder dut (
        .a(a),
        .b(b),
        .cin(cin),
        .sum(sum),
        .cout(cout)
    );

    initial begin
        for (i = 0; i < 8; i = i + 1) begin
            {a, b, cin} = i;
            expected = {1'b0, a}
                     + {1'b0, b}
                     + {1'b0, cin};
            #10;

            if ({cout, sum} !== expected) begin
                $display(
                    "FAILED: a=%b b=%b cin=%b expected=%b actual=%b",
                    a, b, cin, expected, {cout, sum}
                );
                $fatal(1, "Functional mismatch");
            end
        end

        $display("full_adder: all 8 tests passed!");
        $finish;
    end
endmodule
"""

assert not full_adder_tb_path.exists(), "Testbench already exists"
full_adder_tb_path.write_text(full_adder_tb)

print(f"Baseline testbench saved: {full_adder_tb_path}")

Baseline testbench saved: /content/HW_ROME_Coverage/adder/full_addertb.v


In [24]:
half_adder_file = half_adder_out / "half_adder.v"
assert half_adder_file.is_file(), "Missing half_adder RTL"

full_adder_prompt = """
Generate a hierarchical full adder in Verilog-2001.

Exact interface:
module full_adder(
    input wire a,
    input wire b,
    input wire cin,
    output wire sum,
    output wire cout
);

Available previously generated submodule:
module half_adder(
    input wire a,
    input wire b,
    output wire sum,
    output wire carry
);

Requirements:
- Compute a + b + cin, with result {cout, sum}.
- Instantiate exactly two half_adder instances.
- The first half_adder adds a and b.
- The second half_adder adds the first sum and cin.
- Combine the two carry outputs with an OR gate.
- Use combinational logic only.
- Output only full_adder.
- Do not repeat the half_adder definition.
- Do not generate or modify the testbench.
"""

full_adder_out = adder_dir / "full_adder"
full_adder_out.mkdir(parents=True, exist_ok=True)

verilog_loop(
    design_prompt=full_adder_prompt,
    module="full_adder",
    testbench=str(full_adder_tb_path),
    max_iterations=10,
    model_type="ChatGPT",
    model_id=model_choice,
    outdir=str(full_adder_out),
    log=str(full_adder_out / "log.txt"),
    prev_modules=[str(half_adder_file)],
)


========== Iteration 0 / 10 ==========
full_adder: all 8 tests passed!
/content/HW_ROME_Coverage/adder/full_addertb.v:39: $finish called at 80000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  1.4179024696350098
Generation time:  1.4013087749481201
Error handling time:  0.016593217849731445


(np.float64(1.4179024696350098),
 np.float64(1.4013087749481201),
 np.float64(0.016593217849731445))

In [25]:
adder4_tb_path = adder_dir / "adder4tb.v"

adder4_tb = """
`timescale 1ns/1ps

module adder4tb;
    reg [3:0] a;
    reg [3:0] b;
    reg cin;
    wire [3:0] sum;
    wire cout;
    reg [4:0] expected;
    integer i;
    integer j;
    integer k;
    integer test_count;

    adder4 dut (
        .a(a),
        .b(b),
        .cin(cin),
        .sum(sum),
        .cout(cout)
    );

    initial begin
        test_count = 0;

        for (i = 0; i < 16; i = i + 1) begin
            for (j = 0; j < 16; j = j + 1) begin
                for (k = 0; k < 2; k = k + 1) begin
                    a = i;
                    b = j;
                    cin = k;
                    expected = {1'b0, a}
                             + {1'b0, b}
                             + {4'b0000, cin};
                    #10;

                    if ({cout, sum} !== expected) begin
                        $display(
                            "FAILED: a=%d b=%d cin=%b expected=%d actual=%d",
                            a, b, cin, expected, {cout, sum}
                        );
                        $fatal(1, "Functional mismatch");
                    end

                    test_count = test_count + 1;
                end
            end
        end

        $display("adder4: all %0d tests passed!", test_count);
        $finish;
    end
endmodule
"""

assert not adder4_tb_path.exists(), "Testbench already exists"
adder4_tb_path.write_text(adder4_tb)

print(f"Baseline testbench saved: {adder4_tb_path}")

Baseline testbench saved: /content/HW_ROME_Coverage/adder/adder4tb.v


In [26]:
full_adder_file = full_adder_out / "full_adder.v"

assert half_adder_file.is_file(), "Missing half_adder RTL"
assert full_adder_file.is_file(), "Missing full_adder RTL"

adder4_prompt = """
Generate a hierarchical 4-bit unsigned ripple-carry adder
in Verilog-2001.

Exact interface:
module adder4(
    input wire [3:0] a,
    input wire [3:0] b,
    input wire cin,
    output wire [3:0] sum,
    output wire cout
);

Available previously generated submodule:
module full_adder(
    input wire a,
    input wire b,
    input wire cin,
    output wire sum,
    output wire cout
);

Requirements:
- Compute a + b + cin, with the 5-bit result {cout, sum}.
- Instantiate exactly four full_adder instances.
- Connect cin to the least significant full_adder.
- Connect each carry output to the next bit's carry input.
- Connect the most significant carry output to cout.
- Use combinational logic only.
- Do not implement the addition using the + operator.
- Output only adder4.
- Do not repeat any submodule definitions.
- Do not generate or modify the testbench.
"""

adder4_out = adder_dir / "adder4"
adder4_out.mkdir(parents=True, exist_ok=True)

verilog_loop(
    design_prompt=adder4_prompt,
    module="adder4",
    testbench=str(adder4_tb_path),
    max_iterations=10,
    model_type="ChatGPT",
    model_id=model_choice,
    outdir=str(adder4_out),
    log=str(adder4_out / "log.txt"),
    prev_modules=[
        str(half_adder_file),
        str(full_adder_file),
    ],
)


========== Iteration 0 / 10 ==========
adder4: all 512 tests passed!
/content/HW_ROME_Coverage/adder/adder4tb.v:52: $finish called at 5120000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  2.731231212615967
Generation time:  2.7134392261505127
Error handling time:  0.0177915096282959


(np.float64(2.731231212615967),
 np.float64(2.7134392261505127),
 np.float64(0.0177915096282959))

In [27]:
adder8_tb_path = adder_dir / "adder8tb.v"

adder8_tb = """
`timescale 1ns/1ps

module adder8tb;
    reg [7:0] a;
    reg [7:0] b;
    reg cin;
    wire [7:0] sum;
    wire cout;
    integer test_count;

    adder8 dut (
        .a(a),
        .b(b),
        .cin(cin),
        .sum(sum),
        .cout(cout)
    );

    task check;
        input [7:0] test_a;
        input [7:0] test_b;
        input test_cin;
        reg [8:0] expected;
        begin
            a = test_a;
            b = test_b;
            cin = test_cin;

            expected = {1'b0, test_a}
                     + {1'b0, test_b}
                     + {8'b0, test_cin};
            #10;

            if ({cout, sum} !== expected) begin
                $display(
                    "FAILED: a=%h b=%h cin=%b expected=%h actual=%h",
                    a, b, cin, expected, {cout, sum}
                );
                $fatal(1, "Functional mismatch");
            end

            test_count = test_count + 1;
        end
    endtask

    initial begin
        $dumpfile("adder8.vcd");
        $dumpvars(0, adder8tb);

        test_count = 0;

        check(8'h00, 8'h00, 1'b0);
        check(8'h01, 8'h00, 1'b0);
        check(8'h00, 8'h01, 1'b0);
        check(8'h03, 8'h04, 1'b0);
        check(8'h0F, 8'h01, 1'b0);
        check(8'h10, 8'h20, 1'b0);
        check(8'hFF, 8'h01, 1'b0);
        check(8'h00, 8'h00, 1'b1);

        $display("adder8: all %0d tests passed!", test_count);
        $finish;
    end
endmodule
"""

assert not adder8_tb_path.exists(), "Testbench already exists"
adder8_tb_path.write_text(adder8_tb)

print(f"Baseline testbench saved: {adder8_tb_path}")

Baseline testbench saved: /content/HW_ROME_Coverage/adder/adder8tb.v


In [28]:
adder4_file = adder4_out / "adder4.v"

adder_rtl_dependencies = [
    half_adder_file,
    full_adder_file,
    adder4_file,
]

for path in adder_rtl_dependencies:
    assert path.is_file(), f"Missing RTL: {path}"

adder8_prompt = """
Generate a hierarchical 8-bit unsigned ripple-carry adder
in Verilog-2001.

Exact interface:
module adder8(
    input wire [7:0] a,
    input wire [7:0] b,
    input wire cin,
    output wire [7:0] sum,
    output wire cout
);

Available previously generated submodule:
module adder4(
    input wire [3:0] a,
    input wire [3:0] b,
    input wire cin,
    output wire [3:0] sum,
    output wire cout
);

Requirements:
- Compute a + b + cin, with the 9-bit result {cout, sum}.
- Instantiate exactly two adder4 instances.
- The low instance handles a[3:0] and b[3:0].
- Connect cin to the low instance.
- Connect the low instance carry output to the high instance cin.
- The high instance handles a[7:4] and b[7:4].
- Connect the high instance carry output to cout.
- Connect both partial sums to the corresponding sum slices.
- Use combinational logic only.
- Do not implement the addition using the + operator.
- Output only adder8.
- Do not repeat any submodule definitions.
- Do not generate or modify the testbench.
"""

adder8_out = adder_dir / "adder8"
adder8_out.mkdir(parents=True, exist_ok=True)

verilog_loop(
    design_prompt=adder8_prompt,
    module="adder8",
    testbench=str(adder8_tb_path),
    max_iterations=10,
    model_type="ChatGPT",
    model_id=model_choice,
    outdir=str(adder8_out),
    log=str(adder8_out / "log.txt"),
    prev_modules=[str(path) for path in adder_rtl_dependencies],
)


========== Iteration 0 / 10 ==========
VCD info: dumpfile adder8.vcd opened for output.
adder8: all 8 tests passed!
/content/HW_ROME_Coverage/adder/adder8tb.v:63: $finish called at 80000 (1ps)

Testbench ran successfully

========== Feedback Loop Complete ==========
Final status:  Testbench ran successfully
Total time:  2.118851900100708
Generation time:  2.1019301414489746
Error handling time:  0.016921520233154297


(np.float64(2.118851900100708),
 np.float64(2.1019301414489746),
 np.float64(0.016921520233154297))

In [29]:
rtl_files = [
    half_adder_file,
    full_adder_file,
    adder4_file,
    adder8_out / "adder8.v",
]

for path in rtl_files:
    assert path.is_file(), f"Missing RTL: {path}"
    print(f"\n{'=' * 60}")
    print(path.name)
    print("=" * 60)
    print(path.read_text())


half_adder.v
module half_adder(
    input wire a,
    input wire b,
    output wire sum,
    output wire carry
);
    assign sum = a ^ b;
    assign carry = a & b;
endmodule


full_adder.v
module full_adder(
    input wire a,
    input wire b,
    input wire cin,
    output wire sum,
    output wire cout
);

    wire sum1;
    wire carry1;
    wire carry2;

    half_adder ha1 (
        .a(a),
        .b(b),
        .sum(sum1),
        .carry(carry1)
    );

    half_adder ha2 (
        .a(sum1),
        .b(cin),
        .sum(sum),
        .carry(carry2)
    );

    assign cout = carry1 | carry2;

endmodule


adder4.v
module adder4(
    input wire [3:0] a,
    input wire [3:0] b,
    input wire cin,
    output wire [3:0] sum,
    output wire cout
);
    wire c1, c2, c3;

    full_adder fa0 (
        .a(a[0]),
        .b(b[0]),
        .cin(cin),
        .sum(sum[0]),
        .cout(c1)
    );

    full_adder fa1 (
        .a(a[1]),
        .b(b[1]),
        .cin(c1),
        .sum(sum[1]

In [30]:
from pathlib import Path
import subprocess

guide_dir = Path("/content/GUIDE")

if not guide_dir.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/GUIDE-EDA/GUIDE.git",
            str(guide_dir),
        ],
        check=True,
    )

coverage_ref_dir = guide_dir / "colab-scripts" / "coverage_exp"
assert coverage_ref_dir.is_dir(), "Coverage reference directory not found"

print("Coverage reference files:")

for path in sorted(coverage_ref_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(coverage_ref_dir))

Coverage reference files:
.gitignore
Makefile
README.md
covered/doc/covered_user_guide.pdf
slides/coverage_flow_slides.pdf
src/simple_alu.v
student_coverage_demo.ipynb
tb/simple_alu_tb.v


In [31]:
import json

for name in ["README.md", "Makefile"]:
    path = coverage_ref_dir / name
    print(f"\n{'=' * 60}")
    print(name)
    print("=" * 60)
    print(path.read_text())

notebook_path = coverage_ref_dir / "student_coverage_demo.ipynb"
coverage_notebook = json.loads(notebook_path.read_text())

for index, cell in enumerate(coverage_notebook["cells"]):
    if cell["cell_type"] == "code":
        source = "".join(cell.get("source", []))
        print(f"\n--- Reference code cell {index} ---")
        print(source)


README.md
# Simple Icarus Verilog + Covered Example

This example uses a very small 4-bit calculator. There is no clock, reset,
state machine, or memory. The goal is to make the simulation and coverage flow
easy to understand.

## The RTL

The RTL is in `src/simple_alu.v` and has only four signals:

| Signal | Direction | Meaning |
|---|---|---|
| `a` | input | first 4-bit number |
| `b` | input | second 4-bit number |
| `op` | input | chooses the operation |
| `result` | output | 4-bit answer |

`op` is a 2-bit value:

| `op` | Operation |
|---|---|
| `00` | `a + b` (ADD) |
| `01` | `a & b` (AND) |
| `10` | `a | b` (OR) |
| `11` | `a ^ b` (XOR) |

For example, when `a=3`, `b=2`, and `op=00`, the result is 5.

## The testbench

The testbench is in `tb/simple_alu_tb.v`. It performs two tests:

1. ADD: `3 + 2 = 5`
2. AND: `6 & 3 = 2`

OR and XOR are deliberately not tested. This gives the coverage report two
clear missing RTL lines to show.

## Run it

Install `iverilog`, `vvp`, and `co

In [32]:
import subprocess
import shutil

if shutil.which("covered") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(
        ["apt-get", "install", "-y", "covered"],
        check=True,
    )

assert shutil.which("covered"), "Covered installation failed"
print("Covered is ready")

Covered is ready


In [33]:
from pathlib import Path
import hashlib
import json
import re
import shlex
import shutil
import subprocess

adder8_file = adder8_out / "adder8.v"

rtl_files = [
    half_adder_file,
    full_adder_file,
    adder4_file,
    adder8_file,
]

baseline_dir = adder_dir / "coverage" / "baseline"
baseline_dir.mkdir(parents=True, exist_ok=True)

baseline_tb = baseline_dir / "tb_baseline.v"
shutil.copy2(adder8_tb_path, baseline_tb)

rtl_hashes = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in rtl_files
}

(baseline_dir / "rtl_hashes.json").write_text(
    json.dumps(rtl_hashes, indent=2)
)

def run_logged(command, work_dir, log_name):
    command = [str(item) for item in command]
    print("$", shlex.join(command))

    result = subprocess.run(
        command,
        cwd=work_dir,
        capture_output=True,
        text=True,
        timeout=120,
    )

    output = result.stdout + result.stderr
    (work_dir / log_name).write_text(
        "$ " + shlex.join(command) + "\n" + output
    )

    print(output if output else "(no output)")
    assert result.returncode == 0, f"Command failed: {log_name}"
    return output

run_logged(
    [
        "iverilog",
        "-g2001",
        "-s", "adder8tb",
        "-o", baseline_dir / "adder8.vvp",
        *rtl_files,
        baseline_tb,
    ],
    baseline_dir,
    "compile.log",
)

simulation_output = run_logged(
    ["vvp", baseline_dir / "adder8.vvp"],
    baseline_dir,
    "simulation.log",
)

assert "passed!" in simulation_output, "Simulation did not pass"
assert "FAILED" not in simulation_output, "Functional failure"

raw_vcd = baseline_dir / "adder8.vcd"
assert raw_vcd.is_file(), "VCD was not generated"

clean_vcd = baseline_dir / "adder8_clean.vcd"
vcd_text = raw_vcd.read_text()

vcd_text = re.sub(
    r"\$comment\b.*?\$end",
    "",
    vcd_text,
    flags=re.DOTALL,
)

clean_vcd.write_text(vcd_text)

score_command = [
    "covered", "score",
    "-t", "adder8",
    "-i", "adder8tb.dut",
]

for path in rtl_files:
    score_command.extend(["-v", path])

score_command.extend([
    "-vcd", clean_vcd,
    "-o", baseline_dir / "adder8.cdd",
])

run_logged(
    score_command,
    baseline_dir,
    "covered_score.log",
)

for detail_level, report_name in [
    ("s", "coverage_summary.rpt"),
    ("v", "coverage_detail.rpt"),
]:
    command = [
        "covered", "report",
        "-m", "ltc",
        "-d", detail_level,
    ]

    if detail_level == "v":
        command.append("-i")

    command.extend([
        "-o", baseline_dir / report_name,
        baseline_dir / "adder8.cdd",
    ])

    run_logged(
        command,
        baseline_dir,
        f"report_{detail_level}.log",
    )

print("\nBASELINE COVERAGE SUMMARY")
print((baseline_dir / "coverage_summary.rpt").read_text())

$ iverilog -g2001 -s adder8tb -o /content/HW_ROME_Coverage/adder/coverage/baseline/adder8.vvp /content/HW_ROME_Coverage/adder/half_adder/half_adder.v /content/HW_ROME_Coverage/adder/full_adder/full_adder.v /content/HW_ROME_Coverage/adder/adder4/adder4.v /content/HW_ROME_Coverage/adder/adder8/adder8.v /content/HW_ROME_Coverage/adder/coverage/baseline/tb_baseline.v
(no output)
$ vvp /content/HW_ROME_Coverage/adder/coverage/baseline/adder8.vvp
VCD info: dumpfile adder8.vcd opened for output.
adder8: all 8 tests passed!
/content/HW_ROME_Coverage/adder/coverage/baseline/tb_baseline.v:63: $finish called at 80000 (1ps)

$ covered score -t adder8 -i adder8tb.dut -v /content/HW_ROME_Coverage/adder/half_adder/half_adder.v -v /content/HW_ROME_Coverage/adder/full_adder/full_adder.v -v /content/HW_ROME_Coverage/adder/adder4/adder4.v -v /content/HW_ROME_Coverage/adder/adder8/adder8.v -vcd /content/HW_ROME_Coverage/adder/coverage/baseline/adder8_clean.vcd -o /content/HW_ROME_Coverage/adder/coverage/b

In [34]:
history_path = adder_dir / "coverage" / "coverage_history.json"

baseline_record = {
    "iteration": "baseline",
    "test_count": 8,
    "rtl_changed": False,
    "tb_changed": False,
    "simulation_passed": True,
    "aggregation": "module",
    "line": {"hit": 3, "total": 3},
    "toggle_01": {"hit": 46, "total": 56},
    "toggle_10": {"hit": 45, "total": 56},
    "logic": {"hit": 10, "total": 10},
    "report_dir": "baseline",
}

history = (
    json.loads(history_path.read_text())
    if history_path.exists()
    else []
)

if not any(item["iteration"] == "baseline" for item in history):
    history.append(baseline_record)
    history_path.write_text(json.dumps(history, indent=2))

print("Baseline coverage history saved")

Baseline coverage history saved


In [35]:
detail_path = baseline_dir / "coverage_detail.rpt"
detail_text = detail_path.read_text()

print(detail_text)

                             ::::::::::::::::::::::::::::::::::::::::::::::::::
                             ::                                              ::
                             ::  Covered -- Verilog Coverage Verbose Report  ::
                             ::                                              ::
                             ::::::::::::::::::::::::::::::::::::::::::::::::::


~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~   GENERAL INFORMATION   ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
* Report generated from CDD file : /content/HW_ROME_Coverage/adder/coverage/baseline/adder8.cdd

* Reported by                    : Instance

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~

In [37]:
history = json.loads(history_path.read_text())

for item in history:
    if item["iteration"] == "baseline":
        item["instance_metrics"] = {
            "line": {"hit": 40, "total": 40},
            "toggle_01": {"hit": 141, "total": 189},
            "toggle_10": {"hit": 137, "total": 189},
            "logic": {"hit": 98, "total": 136},
        }

history_path.write_text(json.dumps(history, indent=2))
print("Instance-level baseline metrics saved")

Instance-level baseline metrics saved


In [38]:
from openai import OpenAI

refine1_dir = adder_dir / "coverage" / "refine_1"
refine1_dir.mkdir(parents=True, exist_ok=True)

feedback = """
Baseline passes 8 functional tests.

Instance-level coverage:
Line: 40/40
Toggle 0->1: 141/189
Toggle 1->0: 137/189
Logic: 98/136

Specific uncovered targets:
- Top-level b and sum have bits missing both toggle directions.
- Top-level cin has a rising transition but no falling transition.
- carry1 never toggles in several full_adder instances.
- low_adder.fa0.carry2 never toggles.
- Several ha1 instances miss XOR input 11 and AND input 11.
- Several ha1 instances also miss XOR input 01.
- Several ha2 instances miss XOR inputs 01 or 10.

Generate targeted external input sequences that exercise:
- Both directions on every bit of a, b, and sum.
- Carry generation at each bit.
- Carry propagation through each bit and the 4-bit boundary.
- cin rising and falling.
- Both half-adder carry paths and missing XOR combinations.

Use ordered transitions with clearing vectors where useful.
Do not use exhaustive enumeration or random testing.
"""

baseline_text = baseline_tb.read_text()
rtl_context = "\n\n".join(path.read_text() for path in rtl_files)
detail_text = (baseline_dir / "coverage_detail.rpt").read_text()

refinement_prompt = (
    "Strengthen the supplied passing testbench by proposing only "
    "additional targeted test vectors. Keep RTL and all existing "
    "testbench checks unchanged.\n"
    "Return a JSON object with a 'tests' array of 1 to 64 entries. "
    "Each entry must have integer a and b in [0,255], "
    "integer cin in [0,1], and a short reason string. "
    "The tests will be appended in the returned order after the "
    "existing baseline checks using the existing check task.\n\n"
    "COVERAGE FEEDBACK:\n" + feedback
    + "\n\nCURRENT TESTBENCH:\n" + baseline_text
    + "\n\nFIXED RTL:\n" + rtl_context
    + "\n\nACTUAL COVERED REPORT:\n" + detail_text
)

(refine1_dir / "coverage_feedback.txt").write_text(feedback)
(refine1_dir / "llm_prompt.txt").write_text(refinement_prompt)

response = OpenAI().chat.completions.create(
    model=model_choice,
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": "You are a Verilog verification engineer."
        },
        {
            "role": "user",
            "content": refinement_prompt
        },
    ],
)

response_text = response.choices[0].message.content
(refine1_dir / "llm_response.json").write_text(response_text)

payload = json.loads(response_text)
additional_tests = payload["tests"]

assert isinstance(additional_tests, list)
assert 1 <= len(additional_tests) <= 64

for test in additional_tests:
    assert type(test["a"]) is int and 0 <= test["a"] <= 255
    assert type(test["b"]) is int and 0 <= test["b"] <= 255
    assert type(test["cin"]) is int and test["cin"] in (0, 1)
    assert isinstance(test["reason"], str)

print(f"Generated {len(additional_tests)} targeted tests")

for index, test in enumerate(additional_tests, start=1):
    print(
        f"{index:02d}: a={test['a']:02X} "
        f"b={test['b']:02X} cin={test['cin']} "
        f"| {test['reason']}"
    )

Generated 17 targeted tests
01: a=08 b=04 cin=1 | Toggle cin fall and clear carries
02: a=FF b=00 cin=1 | Toggle top-level b and sum
03: a=F0 b=0F cin=0 | Toggle carry propagation across 4-bit boundary
04: a=01 b=01 cin=1 | Test full adder inputs 11, toggle carry paths
05: a=07 b=08 cin=1 | Exercise half-adder XOR input 01
06: a=01 b=03 cin=0 | Toggle carry generation at third bit
07: a=01 b=7F cin=1 | Test propagation through all low bits
08: a=03 b=C0 cin=0 | Toggle ha1 XOR input 01 in high bits
09: a=1F b=E0 cin=0 | Explore mixing patterns for 4-to-4 propagation
10: a=08 b=F9 cin=1 | Trigger specific carry propagations
11: a=40 b=40 cin=1 | Toggle XOR and AND combinations in middle bits
12: a=0E b=0E cin=0 | Toggle sum generation with middle values
13: a=7F b=80 cin=0 | Toggle ultimate carry over through middle bits
14: a=10 b=EF cin=0 | Complete ci0 low carry path
15: a=E1 b=3C cin=1 | Check middle section XOR path variances
16: a=42 b=81 cin=0 | Toggle low half-adder transitions
1

In [39]:
current_hashes = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in rtl_files
}

assert current_hashes == rtl_hashes, "RTL changed since baseline"

new_calls = "\n".join(
    f"        check(8'h{test['a']:02X}, "
    f"8'h{test['b']:02X}, 1'b{test['cin']});"
    for test in additional_tests
)

insertion_point = '        $display("adder8: all %0d tests passed!", test_count);'
assert baseline_text.count(insertion_point) == 1

refined_text = baseline_text.replace(
    insertion_point,
    new_calls + "\n\n" + insertion_point,
    1,
)

refined_tb_path = refine1_dir / "tb_iter_1.v"
refined_tb_path.write_text(refined_text)

assert refined_text.replace(
    new_calls + "\n\n" + insertion_point,
    insertion_point,
    1,
) == baseline_text

print("All baseline tests and checks preserved")
print("RTL unchanged")
print(f"Total tests: {8 + len(additional_tests)}")
print(f"Refined testbench saved: {refined_tb_path}")

All baseline tests and checks preserved
RTL unchanged
Total tests: 25
Refined testbench saved: /content/HW_ROME_Coverage/adder/coverage/refine_1/tb_iter_1.v


In [40]:
current_hashes = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in rtl_files
}
assert current_hashes == rtl_hashes, "RTL changed since baseline"

run_logged(
    [
        "iverilog",
        "-g2001",
        "-s", "adder8tb",
        "-o", refine1_dir / "adder8.vvp",
        *rtl_files,
        refined_tb_path,
    ],
    refine1_dir,
    "compile.log",
)

refined_simulation = run_logged(
    ["vvp", refine1_dir / "adder8.vvp"],
    refine1_dir,
    "simulation.log",
)

assert "passed!" in refined_simulation, "Stop: repair RTL with this TB fixed"
assert "FAILED" not in refined_simulation, "Stop: functional failure"

raw_vcd = refine1_dir / "adder8.vcd"
assert raw_vcd.is_file(), "VCD was not generated"

clean_vcd = refine1_dir / "adder8_clean.vcd"
clean_vcd.write_text(
    re.sub(
        r"\$comment\b.*?\$end",
        "",
        raw_vcd.read_text(),
        flags=re.DOTALL,
    )
)

score_command = [
    "covered", "score",
    "-t", "adder8",
    "-i", "adder8tb.dut",
]

for path in rtl_files:
    score_command.extend(["-v", path])

score_command.extend([
    "-vcd", clean_vcd,
    "-o", refine1_dir / "adder8.cdd",
])

run_logged(score_command, refine1_dir, "covered_score.log")

report_settings = [
    ("coverage_summary.rpt", ["-d", "s"]),
    ("coverage_instance_summary.rpt", ["-d", "s", "-i"]),
    ("coverage_detail.rpt", ["-d", "v", "-i"]),
]

for report_name, options in report_settings:
    run_logged(
        [
            "covered", "report",
            "-m", "ltc",
            *options,
            "-o", refine1_dir / report_name,
            refine1_dir / "adder8.cdd",
        ],
        refine1_dir,
        report_name + ".log",
    )

print("\nREFINE 1: INSTANCE COVERAGE SUMMARY")
print((refine1_dir / "coverage_instance_summary.rpt").read_text())

$ iverilog -g2001 -s adder8tb -o /content/HW_ROME_Coverage/adder/coverage/refine_1/adder8.vvp /content/HW_ROME_Coverage/adder/half_adder/half_adder.v /content/HW_ROME_Coverage/adder/full_adder/full_adder.v /content/HW_ROME_Coverage/adder/adder4/adder4.v /content/HW_ROME_Coverage/adder/adder8/adder8.v /content/HW_ROME_Coverage/adder/coverage/refine_1/tb_iter_1.v
(no output)
$ vvp /content/HW_ROME_Coverage/adder/coverage/refine_1/adder8.vvp
VCD info: dumpfile adder8.vcd opened for output.
adder8: all 25 tests passed!
/content/HW_ROME_Coverage/adder/coverage/refine_1/tb_iter_1.v:81: $finish called at 250000 (1ps)

$ covered score -t adder8 -i adder8tb.dut -v /content/HW_ROME_Coverage/adder/half_adder/half_adder.v -v /content/HW_ROME_Coverage/adder/full_adder/full_adder.v -v /content/HW_ROME_Coverage/adder/adder4/adder4.v -v /content/HW_ROME_Coverage/adder/adder8/adder8.v -vcd /content/HW_ROME_Coverage/adder/coverage/refine_1/adder8_clean.vcd -o /content/HW_ROME_Coverage/adder/coverage/ref

In [41]:
history = json.loads(history_path.read_text())

refine1_record = {
    "iteration": "refine_1",
    "test_count": 25,
    "rtl_changed": False,
    "tb_changed": True,
    "simulation_passed": True,
    "aggregation": "module",
    "instance_metrics": {
        "line": {"hit": 40, "total": 40},
        "toggle_01": {"hit": 185, "total": 189},
        "toggle_10": {"hit": 185, "total": 189},
        "logic": {"hit": 128, "total": 136},
    },
    "report_dir": "refine_1",
}

history = [
    item for item in history
    if item["iteration"] != "refine_1"
]
history.append(refine1_record)

history_path.write_text(json.dumps(history, indent=2))
print("Refine 1 coverage history saved")

Refine 1 coverage history saved


In [43]:
refine2_dir = adder_dir / "coverage" / "refine_2"
refine2_dir.mkdir(parents=True, exist_ok=True)

previous_tb_text = refined_tb_path.read_text()
previous_detail = (refine1_dir / "coverage_detail.rpt").read_text()

feedback2 = """
The current testbench passes all 25 tests.

Instance-level coverage:
Line: 40/40
Toggle 0->1: 185/189
Toggle 1->0: 185/189
Logic: 128/136

Remaining targets:
1. high_adder.fa0.carry1 and high_adder.fa0.ha1.carry:
   both toggle directions are missing.
   Exercise a[4]=b[4]=1, then clear the carry.
   This also targets the missing ha1 XOR/AND input 11
   and the fa0 OR expression's carry1=1 case.

2. high_adder.fa3.carry1 and high_adder.fa3.ha1.carry:
   both toggle directions are missing.
   Exercise a[7]=b[7]=1, then clear the carry.
   This also targets the missing ha1 XOR/AND input 11
   and the fa3 OR expression's carry1=1 case.

3. high_adder.fa1.ha2 XOR input 01 is missing:
   require a[5] XOR b[5] = 0 with carry into bit 5 = 1.

4. high_adder.fa2.ha2 XOR input 01 is missing:
   require a[6] XOR b[6] = 0 with carry into bit 6 = 1.

Propose a short ordered sequence of targeted external inputs.
Include clearing vectors to produce falling transitions.
Do not change RTL, remove tests, or change the checker.
"""

prompt2 = (
    "Return a JSON object with a 'tests' array of 1 to 12 "
    "additional targeted vectors. Each entry must contain "
    "integer a and b in [0,255], integer cin in [0,1], "
    "and a short reason string. These vectors will be appended "
    "in order using the existing check task. "
    "Do not rewrite the testbench. Do not use random or exhaustive tests.\n\n"
    + feedback2
    + "\n\nCURRENT TESTBENCH:\n" + previous_tb_text
    + "\n\nFIXED RTL:\n" + rtl_context
    + "\n\nCOVERED REPORT:\n" + previous_detail
)

(refine2_dir / "coverage_feedback.txt").write_text(feedback2)
(refine2_dir / "llm_prompt.txt").write_text(prompt2)

response2 = OpenAI().chat.completions.create(
    model=model_choice,
    response_format={"type": "json_object"},
    messages=[
        {
            "role": "system",
            "content": "You are a Verilog verification engineer."
        },
        {"role": "user", "content": prompt2},
    ],
)

response2_text = response2.choices[0].message.content
(refine2_dir / "llm_response.json").write_text(response2_text)

tests2 = json.loads(response2_text)["tests"]

assert isinstance(tests2, list)
assert 1 <= len(tests2) <= 12

for test in tests2:
    assert type(test["a"]) is int and 0 <= test["a"] <= 255
    assert type(test["b"]) is int and 0 <= test["b"] <= 255
    assert type(test["cin"]) is int and test["cin"] in (0, 1)
    assert isinstance(test["reason"], str)

assert {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in rtl_files
} == rtl_hashes, "RTL changed since baseline"

calls2 = "\n".join(
    f"        check(8'h{test['a']:02X}, "
    f"8'h{test['b']:02X}, 1'b{test['cin']});"
    for test in tests2
)

assert previous_tb_text.count(insertion_point) == 1

refined2_text = previous_tb_text.replace(
    insertion_point,
    calls2 + "\n\n" + insertion_point,
    1,
)

assert refined2_text.replace(
    calls2 + "\n\n" + insertion_point,
    insertion_point,
    1,
) == previous_tb_text

refined2_tb_path = refine2_dir / "tb_iter_2.v"
refined2_tb_path.write_text(refined2_text)

for index, test in enumerate(tests2, start=1):
    print(
        f"{index:02d}: a={test['a']:02X} "
        f"b={test['b']:02X} cin={test['cin']} "
        f"| {test['reason']}"
    )

print("All previous tests and checks preserved")
print("RTL unchanged")
print(f"Total tests: {25 + len(tests2)}")

01: a=10 b=10 cin=0 | Toggle high_adder.fa0.carry1: set carry a[4]=b[4]=1
02: a=00 b=00 cin=0 | Clear high_adder.fa0.carry1: clear carry a[4]=b[4]=0
03: a=80 b=80 cin=0 | Toggle high_adder.fa3.carry1: set carry a[7]=b[7]=1
04: a=00 b=00 cin=0 | Clear high_adder.fa3.carry1: clear carry a[7]=b[7]=0
05: a=20 b=20 cin=1 | Target high_adder.fa1.ha2 XOR input 01: a[5]=b[5]=0 with carry into bit 5
06: a=40 b=40 cin=1 | Target high_adder.fa2.ha2 XOR input 01: a[6]=b[6]=0 with carry into bit 6
All previous tests and checks preserved
RTL unchanged
Total tests: 31


In [44]:
run_logged(
    [
        "iverilog", "-g2001",
        "-s", "adder8tb",
        "-o", refine2_dir / "adder8.vvp",
        *rtl_files,
        refined2_tb_path,
    ],
    refine2_dir,
    "compile.log",
)

simulation2 = run_logged(
    ["vvp", refine2_dir / "adder8.vvp"],
    refine2_dir,
    "simulation.log",
)

assert "passed!" in simulation2, "Stop: repair RTL with this TB fixed"
assert "FAILED" not in simulation2, "Stop: functional failure"

raw2 = refine2_dir / "adder8.vcd"
assert raw2.is_file(), "VCD was not generated"

clean2 = refine2_dir / "adder8_clean.vcd"
clean2.write_text(
    re.sub(
        r"\$comment\b.*?\$end",
        "",
        raw2.read_text(),
        flags=re.DOTALL,
    )
)

score2 = [
    "covered", "score",
    "-t", "adder8",
    "-i", "adder8tb.dut",
]

for path in rtl_files:
    score2.extend(["-v", path])

score2.extend([
    "-vcd", clean2,
    "-o", refine2_dir / "adder8.cdd",
])

run_logged(score2, refine2_dir, "covered_score.log")

for report_name, options in report_settings:
    run_logged(
        [
            "covered", "report", "-m", "ltc",
            *options,
            "-o", refine2_dir / report_name,
            refine2_dir / "adder8.cdd",
        ],
        refine2_dir,
        report_name + ".log",
    )

print("\nREFINE 2: INSTANCE COVERAGE TOTALS")

report2 = (refine2_dir / "coverage_instance_summary.rpt").read_text()

for line in report2.splitlines():
    if "COVERAGE RESULTS" in line or line.strip().startswith("Accumulated"):
        print(line.strip())

$ iverilog -g2001 -s adder8tb -o /content/HW_ROME_Coverage/adder/coverage/refine_2/adder8.vvp /content/HW_ROME_Coverage/adder/half_adder/half_adder.v /content/HW_ROME_Coverage/adder/full_adder/full_adder.v /content/HW_ROME_Coverage/adder/adder4/adder4.v /content/HW_ROME_Coverage/adder/adder8/adder8.v /content/HW_ROME_Coverage/adder/coverage/refine_2/tb_iter_2.v
(no output)
$ vvp /content/HW_ROME_Coverage/adder/coverage/refine_2/adder8.vvp
VCD info: dumpfile adder8.vcd opened for output.
adder8: all 31 tests passed!
/content/HW_ROME_Coverage/adder/coverage/refine_2/tb_iter_2.v:88: $finish called at 310000 (1ps)

$ covered score -t adder8 -i adder8tb.dut -v /content/HW_ROME_Coverage/adder/half_adder/half_adder.v -v /content/HW_ROME_Coverage/adder/full_adder/full_adder.v -v /content/HW_ROME_Coverage/adder/adder4/adder4.v -v /content/HW_ROME_Coverage/adder/adder8/adder8.v -vcd /content/HW_ROME_Coverage/adder/coverage/refine_2/adder8_clean.vcd -o /content/HW_ROME_Coverage/adder/coverage/ref

In [45]:
final_test_count = 25 + len(tests2)

assert {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in rtl_files
} == rtl_hashes, "RTL changed since baseline"

history = json.loads(history_path.read_text())

refine2_record = {
    "iteration": "refine_2",
    "test_count": final_test_count,
    "rtl_changed": False,
    "tb_changed": True,
    "simulation_passed": True,
    "aggregation": "instance",
    "instance_metrics": {
        "line": {"hit": 40, "total": 40},
        "toggle_01": {"hit": 189, "total": 189},
        "toggle_10": {"hit": 189, "total": 189},
        "logic": {"hit": 136, "total": 136},
    },
    "report_dir": "refine_2",
}

history = [
    item for item in history
    if item["iteration"] != "refine_2"
]
history.append(refine2_record)

history_path.write_text(json.dumps(history, indent=2))

final_dir = adder_dir / "final"
final_dir.mkdir(parents=True, exist_ok=True)

for path in rtl_files:
    shutil.copy2(path, final_dir / path.name)

shutil.copy2(refined2_tb_path, final_dir / "tb_final.v")

prompts = {
    "half_adder": half_adder_prompt,
    "full_adder": full_adder_prompt,
    "adder4": adder4_prompt,
    "adder8": adder8_prompt,
}

prompt_dir = adder_dir / "prompts"
prompt_dir.mkdir(parents=True, exist_ok=True)

for module, prompt in prompts.items():
    (prompt_dir / f"{module}.txt").write_text(prompt)

print(f"Final test count: {final_test_count}")
print("Final RTL and testbench saved")
print("Generation prompts saved")
print("Coverage history saved")
print("RTL unchanged across coverage iterations")

Final test count: 31
Final RTL and testbench saved
Generation prompts saved
Coverage history saved
RTL unchanged across coverage iterations


In [46]:
from google.colab import files

archive = shutil.make_archive(
    "/content/rome_coverage_checkpoint",
    "zip",
    root_dir="/content",
    base_dir="HW_ROME_Coverage",
)

files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>